In [ ]:
import subprocess

def run(command):
    """Run a shell command, show its output live, and fail the notebook if the command fails."""
    print("$", command, flush=True)
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"exit code {process.returncode}: {command}")


In [ ]:
run("nvidia-smi")
run("pip install -q transformers==4.57.1 peft==0.17.1 accelerate")

In [ ]:
%%writefile convert_mapqa.py
#!/usr/bin/env python3
"""Convert MapQA-S into derenderer training files: one map image + the JSON the model should output.

Continuous legend (smooth color bar):
    {"title":"The Percentage of ceremonies in the USA","legend_title":"Value","legend_type":"continuous",
     "data_type":"percentage","values":{"AK":37.9,"AL":39.8,...,"WY":47.4}}

Discrete legend (ranges):
    {"title":"The Number of shots in the USA","legend_title":null,"legend_type":"discrete","data_type":"number",
     "ranges":["7548477-7658039","7668882-7761001","7811458-7859166","7920264-8002936"],
     "values":{"AK":0,"AL":0,...,"WY":1}}

Rules:
- values are keyed by state code, sorted, all 50 states every time
- discrete: each state gets its range number (0 = lowest range); "ranges" lists the legend ranges lowest
  first, whatever order the image shows them in; states with no data get null (the image shows "N/A")
- continuous: numbers are rounded so the color bar has about 100 steps, since the image can't show more
- legend_title is the text above the legend: "Value" on continuous maps, nothing on discrete maps
- data_type is "percentage" or "number": every MapQA title says which, and percentage ranges carry "%"
- train/dev/test follow MapQA's own question files; the few maps that have no questions are skipped

Usage:
    python mapqa/convert_mapqa.py --mapqa ~/Downloads/MapQA_S.zip --out data/mapqa_derender
    python mapqa/convert_mapqa.py --mapqa /kaggle/input/<your-dataset> --out /kaggle/working/mapqa_derender

--mapqa can be the zip or any folder that contains MapQA_S/.

Output: <out>/train.jsonl, dev.jsonl, test.jsonl with one line per map:
    {"id": "map_10", "image": "images/map_10.png", "target": "<the JSON above, as a string>",
     "legend_type": "discrete", "data_type": "number", "true_values": {"AK": 7603168, ...}}
"image" is relative to the MapQA_S folder. true_values are the exact numbers, for evaluation only.
"""

import argparse
import json
import math
import re
import statistics
import zipfile
from pathlib import Path

SPLITS = ("train", "dev", "test")
CONTINUOUS_LEGEND_TITLE = "Value"  # checked on several MapQA-S images; discrete maps show no legend title


class MapQASource:
    """Reads MapQA_S files from the downloaded zip or from an extracted folder."""

    def __init__(self, path):
        path = Path(path).expanduser()
        if path.is_file() and zipfile.is_zipfile(path):
            self.zip = zipfile.ZipFile(path)
            names = [n for n in self.zip.namelist() if not n.startswith("__MACOSX")]
            first_meta = next((n for n in names if re.search(r"(^|/)metadata/map_\d+\.json$", n)), None)
            if first_meta is None:
                raise FileNotFoundError(f"no metadata/map_*.json inside {path}")
            self.prefix = first_meta[: first_meta.index("metadata/")]
            self.names = {n[len(self.prefix):] for n in names if n.startswith(self.prefix)}
        else:
            self.zip = None
            patterns = ["*/" * depth + "metadata" for depth in range(5)]
            root = next((p.parent for pattern in patterns for p in path.glob(pattern)
                         if p.is_dir() and "__MACOSX" not in p.parts), None)
            if root is None:
                raise FileNotFoundError(f"no metadata/ folder found under {path}")
            self.root = root
        self.location = path

    def read(self, rel):
        if self.zip:
            return self.zip.read(self.prefix + rel)
        return (self.root / rel).read_bytes()

    def exists(self, rel):
        return rel in self.names if self.zip else (self.root / rel).exists()

    def metadata_ids(self):
        if self.zip:
            found = (re.fullmatch(r"metadata/map_(\d+)\.json", n) for n in self.names)
        else:
            found = (re.fullmatch(r"map_(\d+)\.json", p.name) for p in (self.root / "metadata").iterdir())
        return sorted(int(m.group(1)) for m in found if m)


def split_of_each_map(source):
    """map id -> split, from the map ids referenced in MapQA's question files."""
    split_of = {}
    for split in SPLITS:
        raw = source.read(f"questions/{split}-QA.json")
        for map_id in re.findall(rb'"map_id"\s*:\s*"map_(\d+)\.png"', raw):
            split_of[int(map_id)] = split
    return split_of


def is_missing(value):
    return value is None or (isinstance(value, float) and math.isnan(value))


def round_for_color_bar(values):
    """Round so the color bar spans about 100 steps: one decimal for a span of ~10, hundreds for ~10,000."""
    present = [v for v in values if v is not None]
    span = max(present) - min(present) if present else 0
    if span <= 0:
        return values
    digits = round(-math.log10(span / 100))
    if digits > 0:
        return [None if v is None else round(v, digits) for v in values]  # never adds digits to the data
    return [None if v is None else int(round(v, digits)) for v in values]


def convert_map(meta):
    """Return (target dict, true values) for one MapQA metadata entry."""
    map_id = meta["map_id"]
    data = sorted(meta["data"], key=lambda d: d["region"])
    if len(data) != 50:
        raise ValueError(f"map {map_id}: expected 50 states, got {len(data)}")
    codes = [d["region"] for d in data]
    true_values = {c: None if is_missing(d["value"]) else d["value"] for c, d in zip(codes, data)}
    title = meta["title"]["text"]
    data_type = meta["data_type"].lower()
    if data_type not in ("percentage", "number"):
        raise ValueError(f"map {map_id}: unknown data_type {meta['data_type']!r}")

    if meta["legend_type"] == "Continuous":
        values = round_for_color_bar([true_values[c] for c in codes])
        target = {"title": title, "legend_title": CONTINUOUS_LEGEND_TITLE, "legend_type": "continuous",
                  "data_type": data_type, "values": dict(zip(codes, values))}
    elif meta["legend_type"] == "Discrete":
        ranges = [c for c in meta["class_descriptions"] if c != "N/A"]
        classes = []
        for d in data:
            if d["class"] < 0:
                if d["description"] != "N/A":
                    raise ValueError(f"map {map_id}: {d['region']} has class {d['class']} but {d['description']!r}")
                classes.append(None)
            else:
                if ranges[d["class"]] != d["description"]:
                    raise ValueError(f"map {map_id}: {d['region']} class {d['class']} != {d['description']!r}")
                classes.append(d["class"])
        target = {"title": title, "legend_title": None, "legend_type": "discrete", "data_type": data_type,
                  "ranges": ranges, "values": dict(zip(codes, classes))}
    else:
        raise ValueError(f"map {map_id}: unknown legend_type {meta['legend_type']!r}")
    return target, true_values


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--mapqa", required=True, help="MapQA_S.zip, or a folder containing MapQA_S/")
    parser.add_argument("--out", type=Path, default=Path("data/mapqa_derender"))
    parser.add_argument("--limit", type=int, help="convert only the first N maps (quick test)")
    args = parser.parse_args()

    source = MapQASource(args.mapqa)
    split_of = split_of_each_map(source)
    ids = source.metadata_ids()[: args.limit]
    args.out.mkdir(parents=True, exist_ok=True)

    files = {split: open(args.out / f"{split}.jsonl", "w", encoding="utf-8") for split in SPLITS}
    counts = {split: {"continuous": 0, "discrete": 0} for split in SPLITS}
    no_split, no_image, lengths = [], [], []
    try:
        for map_id in ids:
            split = split_of.get(map_id)
            if split is None:
                no_split.append(map_id)
                continue
            image = f"images/map_{map_id}.png"
            if not source.exists(image):
                no_image.append(map_id)
                continue
            meta = json.loads(source.read(f"metadata/map_{map_id}.json"))
            target, true_values = convert_map(meta)
            target_text = json.dumps(target, ensure_ascii=False, separators=(",", ":"))
            record = {
                "id": f"map_{map_id}",
                "image": image,
                "target": target_text,
                "legend_type": target["legend_type"],
                "data_type": target["data_type"],
                "true_values": true_values,
            }
            files[split].write(json.dumps(record, ensure_ascii=False) + "\n")
            counts[split][target["legend_type"]] += 1
            lengths.append(len(target_text))
    finally:
        for f in files.values():
            f.close()

    print(f"Read {len(ids)} maps from {source.location}")
    for split in SPLITS:
        c = counts[split]
        print(f"  {split:5} {c['continuous'] + c['discrete']:6} maps "
              f"({c['discrete']} discrete, {c['continuous']} continuous) -> {args.out / (split + '.jsonl')}")
    if no_split:
        print(f"  skipped {len(no_split)} map(s) that no question file uses: {no_split[:10]}")
    if no_image:
        print(f"  skipped {len(no_image)} map(s) with no image: {no_image[:10]}")
    if lengths:
        print(f"  target length in characters: min {min(lengths)}, median {int(statistics.median(lengths))}, "
              f"max {max(lengths)}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile common.py
"""Shared pieces of the derenderer training scripts: loading maps, the training loop, and scoring.

Training data is MapQA (from convert_mapqa.py) plus, with --synth, synthetic maps from
synth/generate_maps.py. Both use the same target JSON.

Scoring (per legend type; MapQA and synthetic test maps are scored separately):
    parsed                 output is a JSON object
    exact_match            output equals the target exactly
    title_correct          title matches (ignoring case and spacing)
    legend_title_correct   legend title matches (ignoring case and spacing; both null also counts)
    legend_type_correct    "continuous" / "discrete" is right
    data_type_correct      "percentage" / "number" is right
    coverage               share of the 50 states present in "values"
    extra_keys             keys in "values" that are not states of this map
  discrete maps:
    ranges_correct         the legend ranges match exactly
    state_accuracy         share of states with the right range index (null for no data)
  continuous maps (errors are relative to the true value range, so 0.05 = 5% of the range):
    nmae, nrmse            mean / root-mean-square error; a missing or non-numeric state counts as 1,
                           and no single error counts more than 1
    within_5pct            share of states within 5% of the range
Maps whose output doesn't parse count as 0 for accuracies and 1 for errors.
"""

import contextlib
import json
import math
import random
import time
from pathlib import Path

import torch
from PIL import Image

def add_common_args(parser):
    parser.add_argument("--data", type=Path, required=True, help="folder with MapQA train.jsonl and test.jsonl")
    parser.add_argument("--mapqa", type=Path, required=True, help="folder containing MapQA_S/images (searched a few levels)")
    parser.add_argument("--synth", type=Path, help="folder containing the synthetic maps (train.jsonl next to "
                                                   "images/synth_*, searched a few levels); adds them to training "
                                                   "and scoring")
    parser.add_argument("--india", type=Path, help="folder containing the synthetic India maps (images/synthind_*); "
                                                   "adds a few to training and scores an India test set")
    parser.add_argument("--india-train-maps", type=int, default=1000, help="India maps mixed into training")
    parser.add_argument("--india-eval-maps", type=int, default=100, help="India test maps to score")
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--train-maps", type=int, help="train on a random subset of this many MapQA maps")
    parser.add_argument("--synth-train-maps", type=int, help="train on a random subset of this many synthetic maps")
    parser.add_argument("--eval-maps", type=int, default=400, help="MapQA test maps to score (same maps for every model)")
    parser.add_argument("--synth-eval-maps", type=int, default=200, help="synthetic test maps to score")
    parser.add_argument("--eval-limit", type=int, help="score only the first N sampled test maps; a prefix of the "
                                                       "usual set, so results stay comparable with earlier runs")
    parser.add_argument("--epochs", type=int, default=2)
    parser.add_argument("--hours", type=float, default=8.0,
                        help="training time budget; the learning-rate schedule is shortened to fit it")
    parser.add_argument("--batch-size", type=int, default=4)
    parser.add_argument("--grad-accum", type=int, default=4)
    parser.add_argument("--lr", type=float, default=5e-5)
    parser.add_argument("--weight-decay", type=float, default=0.01)
    parser.add_argument("--eval-batch-size", type=int, default=8)
    parser.add_argument("--max-new-tokens", type=int, default=640, help="longest MapQA target is about 530 tokens")
    parser.add_argument("--workers", type=int, default=2)
    parser.add_argument("--log-every", type=int, default=20)
    parser.add_argument("--skip-train", action="store_true", help="only score the model as loaded")
    parser.add_argument("--seed", type=int, default=0)


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)


def find_images_root(path, pattern):
    """The folder whose images/ holds files matching pattern, searched a few levels under path.

    Finding datasets by their files rather than folder names works however Kaggle mounts or unpacks them.
    """
    for depth in range(5):
        hits = Path(path).glob("*/" * depth + "images/" + pattern)
        hit = next((p for p in hits if "__MACOSX" not in p.parts), None)
        if hit:
            return hit.parent.parent
    raise FileNotFoundError(f"no images/{pattern} under {path}")


def find_mapqa_root(path):
    return find_images_root(path, "map_*.png")


def find_saved(path, marker):
    """A saved model folder: path itself when it holds marker, else a folder a few levels down that does.

    Lets a previous Kaggle run's output be passed as /kaggle/input. A name that is not a path, such as a
    Hugging Face model id, comes back unchanged.
    """
    path = Path(path)
    if not path.exists():
        return str(path)
    if (path / marker).exists():
        return str(path)
    for depth in range(1, 6):
        hit = next(path.glob("*/" * depth + marker), None)
        if hit:
            return str(hit.parent)
    raise FileNotFoundError(f"no {marker} under {path}")


def read_split(data_dir, split, image_root):
    """Records of one split, each with image_path set (a record's "image" is relative to image_root)."""
    records = []
    with open(Path(data_dir) / f"{split}.jsonl", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            record["image_path"] = str(Path(image_root) / record["image"])
            records.append(record)
    return records


def sample_records(records, n, seed):
    if n is None or n >= len(records):
        return list(records)
    return random.Random(seed).sample(records, n)


def training_records(args, mapqa_root):
    records = [] if args.train_maps == 0 else sample_records(read_split(args.data, "train", mapqa_root),
                                                             args.train_maps, args.seed)
    synthetic = []
    if args.synth:
        folder = find_images_root(args.synth, "synth_*")
        synthetic = sample_records(read_split(folder, "train", folder), args.synth_train_maps, args.seed)
    india = []
    if args.india:
        folder = find_images_root(args.india, "synthind_*")
        india = sample_records(read_split(folder, "train", folder), args.india_train_maps, args.seed)
    print(f"training maps: {len(records)} MapQA + {len(synthetic)} synthetic + {len(india)} India", flush=True)
    return records + synthetic + india


def evaluation_sets(args, mapqa_root):
    """(name, records) pairs: MapQA test maps, plus synthetic US / India test maps when those are given."""
    sets = []
    if args.eval_maps:
        sets.append(("mapqa", sample_records(read_split(args.data, "test", mapqa_root), args.eval_maps, seed=0)))
    if args.synth:
        folder = find_images_root(args.synth, "synth_*")
        sets.append(("synth", sample_records(read_split(folder, "test", folder), args.synth_eval_maps, seed=0)))
    if args.india:
        folder = find_images_root(args.india, "synthind_*")
        sets.append(("india", sample_records(read_split(folder, "test", folder), args.india_eval_maps, seed=0)))
    if args.eval_limit:
        sets = [(name, records[: args.eval_limit]) for name, records in sets]
    return sets


class MapDataset(torch.utils.data.Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        record = self.records[i]
        return Image.open(record["image_path"]).convert("RGB"), record["target"]


def as_list(batch):
    return batch


def autocast(device, amp_dtype):
    return torch.autocast(device.type, dtype=amp_dtype) if amp_dtype else contextlib.nullcontext()


def train_loop(model, loader, args, forward, out_dir, amp_dtype=None, groups=None):
    """AdamW with warmup + cosine decay. Stops after --epochs or --hours, whichever comes first.

    forward(batch) returns the loss. After 20 updates the schedule is shortened, if needed, so the
    learning rate still decays to zero within the time budget. groups: optional AdamW parameter groups,
    for weights that need their own learning rate; the warmup and decay apply to every group alike.
    """
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(groups or params, lr=args.lr, weight_decay=args.weight_decay)
    schedule = {"total": max(1, args.epochs * math.ceil(len(loader) / args.grad_accum))}

    def lr_factor(update):
        warmup = max(1, round(0.03 * schedule["total"]))
        if update < warmup:
            return (update + 1) / warmup
        progress = min(1.0, (update - warmup) / max(1, schedule["total"] - warmup))
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler("cuda", enabled=amp_dtype == torch.float16)
    device = next(model.parameters()).device
    start = time.time()
    deadline = start + args.hours * 3600
    update, update_loss, window = 0, 0.0, []
    model.train()
    with open(out_dir / "train_log.jsonl", "a") as log:
        for epoch in range(args.epochs):
            for i, batch in enumerate(loader, 1):
                with autocast(device, amp_dtype):
                    loss = forward(batch)
                scaler.scale(loss / args.grad_accum).backward()
                update_loss += loss.item() / args.grad_accum
                if i % args.grad_accum and i != len(loader):
                    continue
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update += 1
                window.append(update_loss)
                update_loss = 0.0
                if update == 20:
                    seconds_per_update = (time.time() - start) / update
                    fits = update + int((deadline - time.time()) / seconds_per_update)
                    schedule["total"] = max(update + 1, min(schedule["total"], fits))
                if update == 1 or update % args.log_every == 0:
                    entry = {"epoch": epoch, "update": update, "of": schedule["total"],
                             "loss": round(sum(window) / len(window), 4), "lr": scheduler.get_last_lr()[0],
                             "minutes": round((time.time() - start) / 60, 1)}
                    window.clear()
                    print(json.dumps(entry), flush=True)
                    log.write(json.dumps(entry) + "\n")
                    log.flush()
                if update >= schedule["total"] or time.time() > deadline:
                    print(f"stopping after {update} updates ({(time.time() - start) / 3600:.2f} h)", flush=True)
                    return update
    return update


# ---------------------------------------------------------------- scoring

def parse_prediction(text):
    """The first JSON object in the model output, or None."""
    start = text.find("{")
    if start < 0:
        return None
    try:
        obj, _ = json.JSONDecoder().raw_decode(text[start:])
    except json.JSONDecodeError:
        return None
    return obj if isinstance(obj, dict) else None


def _same_text(a, b):
    if isinstance(a, str) and isinstance(b, str):
        return " ".join(a.split()).lower() == " ".join(b.split()).lower()
    return a is None and b is None


def _is_number(v):
    return isinstance(v, (int, float)) and not isinstance(v, bool) and math.isfinite(v)


def score_map(record, prediction_text):
    target = json.loads(record["target"])
    pred = parse_prediction(prediction_text)
    score = {"id": record["id"], "legend_type": target["legend_type"], "parsed": pred is not None}
    if pred is None:
        return score
    values = pred.get("values") if isinstance(pred.get("values"), dict) else {}
    codes = list(target["values"])
    score["exact_match"] = pred == target
    score["title_correct"] = _same_text(pred.get("title"), target["title"])
    score["legend_title_correct"] = _same_text(pred.get("legend_title"), target["legend_title"])
    score["legend_type_correct"] = pred.get("legend_type") == target["legend_type"]
    score["data_type_correct"] = pred.get("data_type") == target["data_type"]
    score["coverage"] = sum(c in values for c in codes) / len(codes)
    score["extra_keys"] = sum(k not in target["values"] for k in values)
    if target["legend_type"] == "discrete":
        score["ranges_correct"] = pred.get("ranges") == target["ranges"]
        right = {c: bool(c in values and values[c] == target["values"][c]) for c in codes}
        score["state_accuracy"] = sum(right.values()) / len(codes)
        score["per_state"] = right
    else:
        truth = {c: v for c, v in record["true_values"].items() if v is not None}
        span = (max(truth.values()) - min(truth.values())) or 1.0
        per_state = {c: min(1.0, abs(values[c] - v) / span) if _is_number(values.get(c)) else 1.0
                     for c, v in truth.items()}
        errors = list(per_state.values())
        score["nmae"] = sum(errors) / len(errors)
        score["nrmse"] = math.sqrt(sum(e * e for e in errors) / len(errors))
        score["within_5pct"] = sum(e <= 0.05 for e in errors) / len(errors)
        score["per_state"] = per_state
    return score


def summarize(scores):
    summary = {}
    for kind in ("discrete", "continuous"):
        group = [s for s in scores if s["legend_type"] == kind]
        if not group:
            continue

        def mean(key, if_unparsed):
            return round(sum(float(s.get(key, if_unparsed)) for s in group) / len(group), 4)

        result = {"maps": len(group)}
        for key in ("parsed", "exact_match", "title_correct", "legend_title_correct", "legend_type_correct",
                    "data_type_correct", "coverage"):
            result[key] = mean(key, 0)
        result["extra_keys"] = mean("extra_keys", 0)
        if kind == "discrete":
            result.update(ranges_correct=mean("ranges_correct", 0), state_accuracy=mean("state_accuracy", 0))
        else:
            result.update(nmae=mean("nmae", 1), nrmse=mean("nrmse", 1), within_5pct=mean("within_5pct", 0))
        result["per_state"] = per_state_summary(group, unparsed=0.0 if kind == "discrete" else 1.0)
        summary[kind] = result
    return summary


def per_state_summary(group, unparsed):
    """Per-state accuracy (discrete) or mean error (continuous), worst state first.

    Small states are the known weak point, so the run has to report them separately rather than
    leaving them averaged into the 50.
    """
    every_code = {code for score in group for code in score.get("per_state", {})}
    totals = {}
    for score in group:
        per_state = score.get("per_state") or {}
        for code in every_code:
            got = float(per_state[code]) if code in per_state else unparsed
            hit, count = totals.get(code, (0.0, 0))
            totals[code] = (hit + got, count + 1)
    means = {code: round(total / count, 4) for code, (total, count) in totals.items()}
    worst = sorted(means, key=lambda c: means[c], reverse=unparsed == 1.0)
    return {code: means[code] for code in worst}


def evaluate_and_save(records, predictions, out_dir, name):
    """Score one test set; writes predictions_<name>.jsonl and metrics_<name>.json."""
    with open(out_dir / f"raw_predictions_{name}.jsonl", "w", encoding="utf-8") as f:  # kept if scoring fails
        for record, prediction in zip(records, predictions):
            f.write(json.dumps({"id": record["id"], "prediction": prediction}, ensure_ascii=False) + "\n")
    scores = [score_map(r, p) for r, p in zip(records, predictions)]
    with open(out_dir / f"predictions_{name}.jsonl", "w", encoding="utf-8") as f:
        for record, prediction, score in zip(records, predictions, scores):
            f.write(json.dumps({"id": record["id"], "prediction": prediction, "target": record["target"],
                                "score": score}, ensure_ascii=False) + "\n")
    summary = summarize(scores)
    (out_dir / f"metrics_{name}.json").write_text(json.dumps(summary, indent=2))
    print(f"{name} test maps:\n{json.dumps(summary, indent=2)}", flush=True)
    return summary


In [ ]:
%%writefile tiles.py
#!/usr/bin/env python3
"""Choose where to magnify a map, from the picture alone.

Small regions are where the derenderer fails: a state a few pixels wide is smaller than one 28x28-pixel
vision token, so its colour is averaged with its neighbours before the model sees it. Rather than naming
the crowded corner by hand (which would only ever be right for one country), this picks the tiles with the
densest region boundaries and lets the model magnify those.

    boxes = busiest_tiles(image, tiles=2)     # boxes in the image's own pixels, biggest crowding first

The legend is full of edges too, so boundaries are only counted inside the map body: the largest connected
patch of non-background pixels, which is the mainland because its states all touch.
"""

import numpy as np
from scipy import ndimage

TILE_FRACTION = 0.16          # a tile covers this share of the map body's width
STRIDE_FRACTION = 0.5         # windows step by half a tile
TOKEN_PIXELS = 28 * 28        # one vision token covers a 28x28-pixel square of the map
SPECK_PIXELS = 40             # smaller than this is a rendering speck between borders, not a region


def map_body(image):
    """Mask of the mainland: the largest connected patch that is not the background colour."""
    pixels = np.asarray(image.convert("RGB"), dtype=np.int16)
    corners = np.stack([pixels[0, 0], pixels[0, -1], pixels[-1, 0], pixels[-1, -1]])
    background = np.median(corners, axis=0)
    filled = np.abs(pixels - background).max(axis=2) > 24
    labels, count = ndimage.label(filled)
    if count == 0:
        return np.ones(filled.shape, dtype=bool)
    sizes = ndimage.sum_labels(filled, labels, index=range(1, count + 1))
    return labels == (int(np.argmax(sizes)) + 1)


def boundary_mask(image, body):
    """Where region borders run: colour changes inside the map body."""
    grey = np.asarray(image.convert("L"), dtype=np.float32)
    steps = np.zeros_like(grey)
    steps[:-1, :-1] = np.abs(np.diff(grey, axis=0)[:, :-1]) + np.abs(np.diff(grey, axis=1)[:-1, :])
    return (steps > 20) & body


def small_regions(image, body, boundaries):
    """Centres of the regions that fit inside a single vision token, where this model loses colours.

    Regions are the patches left when the boundaries are removed. What matters is not how many borders a
    place has but how many of its regions are too small for the tower to resolve, so those are counted
    directly. Anything under a few dozen pixels is a speck where two borders meet, not a region.
    The pixel thresholds assume the map reaches the tower near its own resolution, which holds for maps
    of up to about a megapixel.
    """
    labels, count = ndimage.label(body & ~boundaries)
    if count == 0:
        return np.zeros((0, 2)), np.zeros(0)
    index = range(1, count + 1)
    areas = np.asarray(ndimage.sum_labels(body, labels, index=index))
    keep = (areas >= SPECK_PIXELS) & (areas < TOKEN_PIXELS)
    if not keep.any():
        return np.zeros((0, 2)), np.zeros(0)
    centres = np.asarray(ndimage.center_of_mass(body, labels, index=[i for i, k in zip(index, keep) if k]))
    return centres.reshape(-1, 2), areas[keep]


def busiest_tiles(image, tiles=2, tile_fraction=TILE_FRACTION):
    """Boxes over the parts of the map holding the most small regions, richest first, non-overlapping.

    Ties, and maps whose regions are all large, fall back to plain boundary density.
    """
    body = map_body(image)
    rows, columns = np.where(body)
    if rows.size == 0:
        return []
    top, bottom, left, right = rows.min(), rows.max(), columns.min(), columns.max()
    size = max(28, int((right - left) * tile_fraction))
    stride = max(7, int(size * STRIDE_FRACTION))
    boundaries = boundary_mask(image, body)
    integral = boundaries.cumsum(0).cumsum(1)
    centres, _ = small_regions(image, body, boundaries)

    def density(y, x):
        y2, x2 = y + size, x + size
        return (integral[y2, x2] - integral[y, x2] - integral[y2, x] + integral[y, x])

    def score(y, x):
        inside = 0
        if len(centres):
            inside = int(((centres[:, 0] >= y) & (centres[:, 0] < y + size) &
                          (centres[:, 1] >= x) & (centres[:, 1] < x + size)).sum())
        return inside * 1000 + density(y, x) / max(1.0, integral[-1, -1]) * 100

    windows = [(score(y, x), x, y)
               for y in range(top, max(top + 1, bottom - size), stride)
               for x in range(left, max(left + 1, right - size), stride)
               if y + size < integral.shape[0] and x + size < integral.shape[1]]
    chosen = []
    for score, x, y in sorted(windows, reverse=True):
        if all(abs(x - bx) >= size * 0.6 or abs(y - by) >= size * 0.6 for bx, by, _, _ in chosen):
            chosen.append((x, y, x + size, y + size))
        if len(chosen) == tiles:
            break
    return chosen


In [ ]:
%%writefile sheet.py
"""Turn a map into a region sheet: one equal cell per region, labelled with its code, beside the map's legend.

A sheet asks the model only to match cells to the legend, whatever the country, so it should need no geography.
Cells are two tokens tall and line up with the token grid: the code fills the top token, and the region's own
interior pixels fill the bottom one, so a tiny region gets a clean token of its own. Regions come from the drawing,
so this tests the sheet with perfect regions; synth/region_labels.py writes what it needs.

In training each sheet shows a random set of regions under random two-letter codes. The only way to answer is to
copy each code from its cell, which is what lets a model that has only seen US maps write Indian codes.
"""

import json
import random
import string
from pathlib import Path

import cv2
import numpy as np
import torch
from matplotlib import font_manager
from PIL import Image, ImageDraw, ImageFont

CELL = 56  # two tokens tall and wide
COLUMNS = 8
GAP = 28  # one empty token between the cells and the legend
LEGEND_BANDS = {"right": (0.75, 0.0, 1.0, 1.0), "bottom": (0.0, 0.82, 1.0, 1.0)}  # draw_map's layout, as shares
SHEET_PROMPT = (
    "The first image is a choropleth map. The second image is a sheet of its regions: each cell shows a region's code "
    "above that region's colour, beside the map's legend. Return only JSON with these keys: "
    '"title"; "legend_title" (null if there is none); "legend_type" ("continuous" or "discrete"); '
    '"data_type" ("percentage" or "number"); "ranges" (discrete legends only: the legend ranges, lowest first); '
    '"values" (every region on the sheet by the code printed in its cell, in sheet order: a number for a continuous '
    "legend, the index of the region's range for a discrete legend, null for no data)."
)


def dominant(pixels):
    """The pixels near the region's commonest colour, dropping printed text and leftover border."""
    if not len(pixels):
        return pixels
    coarse = pixels.astype(int) // 16
    colours, counts = np.unique(coarse, axis=0, return_counts=True)
    return pixels[np.abs(coarse - colours[np.argmax(counts)]).max(1) <= 1]


class Sheets:
    def __init__(self, labels):
        """labels: a folder holding regions.json / .png (India) and regions_usa.json / .png, searched a few levels."""
        folder = next(Path(labels).rglob("regions_usa.json")).parent
        self.maps = {}
        for prefix, name in (("synthind_", "regions"), ("synth_", "regions_usa")):
            self.maps[prefix] = (json.loads((folder / f"{name}.json").read_text()),
                                 np.asarray(Image.open(folder / f"{name}.png")))
        self.typeface = ImageFont.truetype(font_manager.findfont("DejaVu Sans"), 16)

    def lookup(self, record):
        return next(v for prefix, v in self.maps.items() if record["id"].startswith(prefix))

    def region_pixels(self, image, record, codes):
        """{code: (n, 3) array of the pixels inside that region, away from its border where the region allows."""
        meta, raster = self.lookup(record)
        left, top, right, bottom = meta["placed"][record["id"]]
        place = np.array([[(right - left) / raster.shape[1], 0, left], [0, (bottom - top) / raster.shape[0], top]])
        drawn = cv2.warpAffine(raster, place, image.size, flags=cv2.INTER_NEAREST)
        pixels, found = np.asarray(image), {}
        for code in codes:
            mask = (drawn == meta["codes"].index(code) + 1).astype(np.uint8)
            chosen = mask
            for size in (3, 5):
                inner = cv2.erode(mask, np.ones((size, size), np.uint8))
                if inner.sum() >= 6:
                    chosen = inner
            found[code] = dominant(pixels[chosen > 0])
        return found

    def legend(self, image, record):
        """The legend, cut from the band the generator keeps for it and trimmed to its contents."""
        x0, y0, x1, y1 = LEGEND_BANDS[record["style"]["legend_pos"]]
        band = image.crop((int(x0 * image.width), int(y0 * image.height), int(x1 * image.width), int(y1 * image.height)))
        pixels = np.asarray(band).astype(int)
        ink = np.abs(pixels - pixels[0, 0]).sum(2) > 30
        rows, columns = np.where(ink)
        return band.crop((columns.min(), rows.min(), columns.max() + 1, rows.max() + 1)) if len(rows) else band

    def draw(self, cells, legend):
        """cells: (code, pixels) in sheet order. Sides are multiples of the token, so nothing is resampled."""
        rows = int(np.ceil(len(cells) / COLUMNS))
        grid_height = rows * CELL
        width = COLUMNS * CELL + GAP + legend.width
        height = max(grid_height, legend.height)
        sheet = Image.new("RGB", (int(np.ceil(width / 28)) * 28, int(np.ceil(height / 28)) * 28), "white")
        draw = ImageDraw.Draw(sheet)
        half = CELL // 2
        for n, (code, pixels) in enumerate(cells):
            x, y = (n % COLUMNS) * CELL, (n // COLUMNS) * CELL
            draw.text((x + half, y + half // 2), code, fill="black", font=self.typeface, anchor="mm")
            fill = np.resize(pixels.astype(np.uint8), (half, CELL, 3)) if len(pixels) else np.full((half, CELL, 3), 255, np.uint8)
            sheet.paste(Image.fromarray(fill), (x, y + half))
        sheet.paste(legend, (COLUMNS * CELL + GAP, 0))
        return sheet

    def scored(self, image, record):
        """A test map's sheet: every listed region under its real code. The target is unchanged."""
        codes = sorted(json.loads(record["target"])["values"])
        found = self.region_pixels(image, record, codes)
        return self.draw([(c, found[c]) for c in codes], self.legend(image, record))

    def taught(self, image, record, rng):
        """A training map's sheet and target: a random set of regions under random codes, in code order."""
        target = json.loads(record["target"])
        real = list(target["values"])
        chosen = rng.sample(real, rng.randint(16, len(real)))
        names = rng.sample([a + b for a in string.ascii_uppercase for b in string.ascii_uppercase], len(chosen))
        renamed = dict(sorted(zip(names, chosen)))
        found = self.region_pixels(image, record, chosen)
        sheet = self.draw([(name, found[code]) for name, code in renamed.items()], self.legend(image, record))
        target["values"] = {name: target["values"][code] for name, code in renamed.items()}
        return sheet, json.dumps(target, ensure_ascii=False, separators=(",", ":"))


class SheetDataset(torch.utils.data.Dataset):
    """(images, boxes, target, fuse, prompt), as CropDataset gives TrainCollate."""

    def __init__(self, records, sheets):
        self.records, self.sheets = records, sheets

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        record = self.records[i]
        image = Image.open(record["image_path"]).convert("RGB")
        sheet, target = self.sheets.taught(image, record, random.Random())
        return [image, sheet], None, target, False, SHEET_PROMPT


In [ ]:
%%writefile train_qwen.py
#!/usr/bin/env python3
"""LoRA fine-tune Qwen2.5-VL (3B by default) to turn a choropleth map into the target JSON, then score it.

    python train_qwen.py --data /kaggle/working/data --mapqa /kaggle/input --out /kaggle/working/qwen
    python train_qwen.py ... --synth /kaggle/input            # also train and score on synthetic maps
    python train_qwen.py ... --skip-train                     # score the base model zero-shot, same prompt
    python train_qwen.py ... --adapter <out>/adapter          # continue training, or score a saved adapter

Writes <out>/adapter (LoRA weights), train_log.jsonl, and predictions_<set>.jsonl + metrics_<set>.json for
each test set (mapqa, plus synth when --synth is given).
"""

import argparse
import json
import math
import os
import random
import time
from pathlib import Path

# Set before torch initialises CUDA. Two images per example leave a lot of reserved-but-unused memory on a
# T4; letting the allocator grow its segments avoids failing on fragmentation alone.
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from peft import LoraConfig, PeftModel, get_peft_model, set_peft_model_state_dict
from PIL import Image
from torch.utils.data import DataLoader
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

from common import (add_common_args, autocast, evaluate_and_save, evaluation_sets, find_mapqa_root,
                    find_saved, read_split, sample_records, set_seed, train_loop, training_records)


def training_records_mapqa(args, root):
    """MapQA training maps only: what the picker is taught on (the rule's labels are tuned to these)."""
    return read_split(args.data, "train", root)
from sheet import SHEET_PROMPT, SheetDataset, Sheets
from tiles import boundary_mask, busiest_tiles, map_body, small_regions

PROMPT = (
    "Read this choropleth map and return only JSON with these keys: "
    '"title"; "legend_title" (null if there is none); "legend_type" ("continuous" or "discrete"); '
    '"data_type" ("percentage" or "number"); '
    '"ranges" (discrete legends only: the legend ranges, lowest first); '
    '"values" (all 50 US states by state code: a number for a continuous legend, '
    "the index of the state's range for a discrete legend, null for no data)."
)
# India maps list 32 states and union territories; the four smallest UTs are drawn but not listed (see
# synth/generate_maps.py). Everything else about the task is the same.
INDIA_PROMPT = PROMPT.replace("all 50 US states by state code",
                              "all 32 Indian states and union territories by state code")
assert INDIA_PROMPT != PROMPT


def prompt_for(record):
    """prompt_note: an extra sentence some probe maps carry, e.g. naming the region outlined on the map."""
    return (INDIA_PROMPT if record.get("id", "").startswith("synthind_") else PROMPT) + record.get("prompt_note", "")


CROP_NOTE = (" The second image is a magnified view of part of the same map; "
             "use it to read the small regions it contains.")
TILES_NOTE = (" The extra images are magnified views of the parts of the same map holding the smallest "
              "regions; use them to read the regions they contain. Each one covers this part of the map, "
              "as a fraction of its width and height:")


def tile_images(image, count, zoom, rng=None, jitter=0, boxes=None):
    """Magnified views of wherever this map's region boundaries are densest.

    The boxes come from the picture alone (train/tiles.py), so nothing here knows which country it is
    looking at - the same rule finds the north-east of a US map and the small states of any other.
    """
    picked = busiest_tiles(image, tiles=count) if boxes is None else boxes
    views, boxes = [], []
    for left, top, right, bottom in picked:
        if jitter and rng is not None:
            shift_x, shift_y = rng.randint(-jitter, jitter), rng.randint(-jitter, jitter)
            left, right = max(0, left + shift_x), min(image.width, right + shift_x)
            top, bottom = max(0, top + shift_y), min(image.height, bottom + shift_y)
        view = image.crop((left, top, right, bottom))
        views.append(view.resize((max(28, round(view.width * zoom)), max(28, round(view.height * zoom))),
                                 Image.BICUBIC))
        boxes.append((left / image.width, top / image.height, right / image.width, bottom / image.height))
    return views, boxes
# Matches the attention and MLP layers of the language model, and the MLP layers of the vision encoder.
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
# Adds the vision encoder's attention and its patch merger. Qwen2.5-VL fuses vision attention into "qkv"
# and "proj", which the names above do not match, so the vision tower's attention is otherwise never trained.
VISION_LORA_REGEX = (r"(.*\.(?:q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj|qkv))"
                     r"|(.*\.attn\.proj)|(.*visual\.merger\.mlp\.[02])")


def resized_size(width, height, factor, min_pixels, max_pixels):
    """The size Qwen's processor resizes an image to: each side a multiple of factor, area within bounds."""
    height_out = max(factor, round(height / factor) * factor)
    width_out = max(factor, round(width / factor) * factor)
    if height_out * width_out > max_pixels:
        shrink = math.sqrt(height * width / max_pixels)
        height_out = max(factor, math.floor(height / shrink / factor) * factor)
        width_out = max(factor, math.floor(width / shrink / factor) * factor)
    elif height_out * width_out < min_pixels:
        grow = math.sqrt(min_pixels / (height * width))
        height_out = math.ceil(height * grow / factor) * factor
        width_out = math.ceil(width * grow / factor) * factor
    return width_out, height_out


def two_scale_images(processor, images, scale):
    """Each image at the size the tower normally sees, and the same image `scale` times larger."""
    image_processor = processor.image_processor
    factor = image_processor.patch_size * image_processor.merge_size
    coarse, fine = [], []
    for image in images:
        width, height = resized_size(image.width, image.height, factor,
                                     image_processor.min_pixels, image_processor.max_pixels)
        coarse.append(image.resize((width, height), Image.BICUBIC))
        fine.append(image.resize((width * scale, height * scale), Image.BICUBIC))
    return coarse, fine


def pool_to_coarse(fine, fine_grid, coarse_grid, merge, scale):
    """Average the fine pass's tokens over the area of each coarse token, image by image."""
    pooled, at = [], 0
    for (frames, fine_h, fine_w), (_, coarse_h, coarse_w) in zip(fine_grid.tolist(), coarse_grid.tolist()):
        rows, columns = coarse_h // merge, coarse_w // merge
        count = frames * rows * scale * columns * scale
        block = fine[at: at + count].view(frames, rows, scale, columns, scale, -1)
        at += count
        pooled.append(block.mean(dim=(2, 4)).reshape(frames * rows * columns, -1))
    return torch.cat(pooled)


class S2Visual(torch.nn.Module):
    """The vision tower run at two scales and fused - "scaling on scales".

    The second pass sees the map enlarged, so a state covering a fraction of one 28x28-pixel token in the
    first pass covers several tokens here. Those fine tokens are averaged back onto the coarse grid and
    concatenated with it, so the language model still receives exactly as many tokens as before; each one
    now also carries what its own patch of map looks like close up.

    The fusion starts as a pass-through of the coarse pass (the fine half of its weights is zero), so a
    model trained without S2 behaves identically until training moves those weights.
    """

    def __init__(self, visual, scale):
        super().__init__()
        self.visual = visual
        self.scale = scale
        width = visual.config.out_hidden_size
        self.fuse = torch.nn.Linear(2 * width, width, bias=False)
        with torch.no_grad():
            self.fuse.weight.zero_()
            self.fuse.weight[:, :width].copy_(torch.eye(width))
        self.pending = None

    def __getattr__(self, name):  # dtype, spatial_merge_size, config and friends come from the tower
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self._modules["visual"], name)

    def forward(self, hidden_states, grid_thw, **kwargs):
        coarse = self.visual(hidden_states, grid_thw=grid_thw, **kwargs)
        pending, self.pending = self.pending, None
        if pending is None:
            return coarse
        pixels, fine_grid = pending
        with torch.no_grad():  # the tower is frozen here; only the fusion learns
            fine = self.visual(pixels.type(self.visual.dtype), grid_thw=fine_grid, **kwargs)
        pooled = pool_to_coarse(fine, fine_grid, grid_thw, self.visual.spatial_merge_size, self.scale)
        return self.fuse(torch.cat([coarse, pooled.to(coarse.dtype)], dim=-1))


def attach_s2(model, scale, device):
    """Replace the vision tower with the two-scale version, wherever it sits in the wrapped model."""
    for module in model.modules():
        if "visual" in module._modules and not isinstance(module._modules["visual"], S2Visual):
            wrapper = S2Visual(module._modules["visual"], scale)
            wrapper.fuse.to(device=device, dtype=torch.float32).requires_grad_(True)
            module._modules["visual"] = wrapper
            return wrapper
    raise SystemExit("could not find the vision tower to attach S2 to")

def fusion_tiles(image, processor, count, cells, zoom):
    """The busiest tiles, snapped to whole cells of the coarse token grid and cut out at `zoom` times size.

    Each tile covers exactly cells x cells coarse tokens, so its zoom x zoom fine tokens per cell line up
    with the coarse grid without any resampling: the tile's tokens can be added to the coarse tokens that
    already sit at those map positions.
    """
    reader = processor.image_processor
    factor = reader.patch_size * reader.merge_size
    width, height = resized_size(image.width, image.height, factor, reader.min_pixels, reader.max_pixels)
    columns, rows = width // factor, height // factor
    scale_x, scale_y = width / image.width, height / image.height
    placed = []
    for left, top, right, bottom in busiest_tiles(image, tiles=count):
        first_column = min(max(0, round((left + right) / 2 * scale_x / factor - cells / 2)), max(0, columns - cells))
        first_row = min(max(0, round((top + bottom) / 2 * scale_y / factor - cells / 2)), max(0, rows - cells))
        box = (first_column * factor / scale_x, first_row * factor / scale_y,
               (first_column + cells) * factor / scale_x, (first_row + cells) * factor / scale_y)
        side = cells * factor * zoom
        placed.append((first_row, first_column, image.resize((side, side), Image.BICUBIC, box=box)))
    return placed


def fusion_inputs(processor, maps, count, cells, zoom):
    """Pixels for every selected tile in a batch, and where each goes: (batch index, first row, first column)."""
    tiles, where = [], []
    for index, image in maps:
        for row, column, tile in fusion_tiles(image, processor, count, cells, zoom):
            tiles.append(tile)
            where.append((index, row, column))
    if not tiles:
        return {}
    seen = processor.image_processor(images=tiles, return_tensors="pt")
    return {"fx_pixel_values": seen["pixel_values"], "fx_grid_thw": seen["image_grid_thw"],
            "fx_where": torch.tensor(where)}


class FusionVisual(torch.nn.Module):
    """The vision tower, plus close-up detail added to the coarse tokens that sit on the busiest tiles.

    A tile is cut from the map, enlarged `zoom` times and run through the same frozen tower. Its tokens
    come back on a grid `zoom` times finer than the coarse one, so each coarse token has zoom x zoom fine
    tokens over exactly the same piece of map. Those are not averaged (averaging is how a state covering
    one fine token gets diluted to a ninth): each is shrunk, the block is laid side by side, and one
    linear layer maps the block to a correction that is added to the coarse token.

    Every other coarse token is untouched, and the language model still receives exactly the tokens it did
    before. The output layer starts at zero, so an untrained fusion changes nothing.
    """

    def __init__(self, visual, cells, zoom, reduced=256, init_std=0.0):
        super().__init__()
        self.visual, self.cells, self.zoom = visual, cells, zoom
        width = visual.config.out_hidden_size
        self.reduce = torch.nn.Linear(width, reduced)
        self.out = torch.nn.Linear(zoom * zoom * reduced, width, bias=False)
        with torch.no_grad():
            # Exactly zero keeps an untrained fusion identical to the baseline, but it also starves the
            # first layer: its gradient passes through this one. A small init_std lets both learn at once.
            if init_std:
                self.out.weight.normal_(0.0, init_std)
            else:
                self.out.weight.zero_()
        self.pending = None

    def __getattr__(self, name):  # dtype, spatial_merge_size, config and friends come from the tower
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self._modules["visual"], name)

    def forward(self, hidden_states, grid_thw, **kwargs):
        coarse = self.visual(hidden_states, grid_thw=grid_thw, **kwargs)
        pending, self.pending = self.pending, None
        if pending is None:
            return coarse
        pixels, tile_grid, where = pending
        with torch.no_grad():  # the tower is frozen here; only the fusion learns
            fine = self.visual(pixels.type(self.visual.dtype), grid_thw=tile_grid, **kwargs)
        cells, zoom = self.cells, self.zoom
        tiles = len(where)
        shrunk = self.reduce(fine.float()).view(tiles, cells, zoom, cells, zoom, -1)
        blocks = shrunk.permute(0, 1, 3, 2, 4, 5).reshape(tiles, cells * cells, -1)
        added = self.out(blocks)                                       # (tiles, cells * cells, width)
        merge = self.visual.spatial_merge_size
        rows_columns = [(h // merge, w // merge) for _, h, w in grid_thw.tolist()]
        offsets = [0]
        for rows, columns in rows_columns:
            offsets.append(offsets[-1] + rows * columns)
        delta = torch.zeros(coarse.shape, dtype=added.dtype, device=coarse.device)
        steps = torch.arange(cells, device=coarse.device)
        for k, (image, row, column) in enumerate(where.tolist()):
            index = offsets[image] + (row + steps)[:, None] * rows_columns[image][1] + (column + steps)[None, :]
            delta = delta.index_add(0, index.reshape(-1), added[k])
        return coarse + delta.to(coarse.dtype)


def attach_fusion(model, cells, zoom, device, init_std=0.0):
    """Replace the vision tower with the fusing version, wherever it sits in the wrapped model."""
    for module in model.modules():
        if "visual" in module._modules and not isinstance(module._modules["visual"], (S2Visual, FusionVisual)):
            wrapper = FusionVisual(module._modules["visual"], cells, zoom, init_std=init_std)
            wrapper.reduce.to(device=device, dtype=torch.float32).requires_grad_(True)
            wrapper.out.to(device=device, dtype=torch.float32).requires_grad_(True)
            module._modules["visual"] = wrapper
            return wrapper
    raise SystemExit("could not find the vision tower to attach the fusion to")


class TokenPicker(torch.nn.Module):
    """A small MLP that reads each of the map's vision tokens and predicts how many regions too small to read
    are centred in that token's square of the map.

    It is trained on counts from the rule in train/tiles.py. The count matters, not just yes/no: the
    north-east packs eight small states into a handful of tokens, and a yes/no label makes that corner look
    no busier than a lake shore. At test time the model picks its own places to magnify from the tokens it
    has already computed, with no image-processing code in the loop.
    """

    def __init__(self, width, hidden=256):
        super().__init__()
        self.net = torch.nn.Sequential(torch.nn.Linear(width, hidden), torch.nn.GELU(), torch.nn.Linear(hidden, 1))

    def forward(self, tokens, shape=None):
        return self.net(tokens).squeeze(-1)  # log of the expected count; every token is judged alone


class ConvPicker(torch.nn.Module):
    """The same job as TokenPicker, but each token is judged together with its neighbours.

    The tokens are laid back on the map's grid; one 1x1 layer shrinks them, then two 3x3 layers (the second
    skipping every other token) let a token see about 7x7 tokens around it. A small state next to other small
    states should read as a cluster, which a token-by-token network cannot see. About as many weights as the
    MLP (0.56M against 0.52M), so a difference is not just size.
    """

    def __init__(self, width, hidden=128):
        super().__init__()
        self.reduce = torch.nn.Conv2d(width, hidden, 1)
        self.context = torch.nn.Sequential(
            torch.nn.GELU(), torch.nn.Conv2d(hidden, hidden, 3, padding=1),
            torch.nn.GELU(), torch.nn.Conv2d(hidden, hidden, 3, padding=2, dilation=2),
            torch.nn.GELU(), torch.nn.Conv2d(hidden, 1, 1))

    def forward(self, tokens, shape):
        rows, columns = shape
        grid = tokens.reshape(-1, rows, columns, tokens.shape[-1]).permute(0, 3, 1, 2)  # maps, width, rows, columns
        return self.context(self.reduce(grid)).reshape(-1)  # log of the expected count, map by map, row by row


PICKERS = {"mlp": TokenPicker, "cnn": ConvPicker}


PICKER_TILE_FRACTION = 0.14  # tile side as a share of image width (the rule's run 0.10-0.15 on MapQA)


def vision_tower(model):
    for module in model.modules():
        if "visual" in module._modules:
            return module._modules["visual"]
    raise SystemExit("could not find the vision tower")


def map_tokens(model, processor, image, device, amp_dtype):
    """The tower's tokens for one map, and the (rows, columns) of the token grid they lie on, row by row."""
    tower = vision_tower(model)
    seen = processor.image_processor(images=[image], return_tensors="pt")
    _, height, width = seen["image_grid_thw"][0].tolist()
    merge = processor.image_processor.merge_size
    with torch.no_grad(), autocast(device, amp_dtype):
        tokens = tower(seen["pixel_values"].to(device).type(tower.dtype), grid_thw=seen["image_grid_thw"].to(device))
    return tokens.float(), (height // merge, width // merge)


def token_counts(image, rows, columns):
    """For each token, how many regions smaller than one token have their centre in its square of the map."""
    body = map_body(image)
    centres, _ = small_regions(image, body, boundary_mask(image, body))
    counts = torch.zeros(rows, columns)
    for y, x in centres:
        counts[min(rows - 1, int(y / image.height * rows)), min(columns - 1, int(x / image.width * columns))] += 1
    return counts.flatten()


def picker_boxes(counts, rows, columns, image, count):
    """Windows scored by the predicted counts inside them, best first, with the rule's overlap test.

    A token counts in proportion to how much of its square the window covers, and windows step by an
    eighth of a tile: counting a token only when its centre falls inside, on a half-tile grid, left most
    windows tied and the tiles landed by accident. With the rule's own counts fed in, these tiles hold
    78.4% of a map's small regions against the rule's 78.6% (200 MapQA test maps).
    """
    grid = counts.view(rows, columns).double()
    size = max(28, int(image.width * PICKER_TILE_FRACTION))
    stride = max(4, size // 8)
    cell_w, cell_h = image.width / columns, image.height / rows
    left_edges, top_edges = torch.arange(columns) * cell_w, torch.arange(rows) * cell_h
    windows = []
    for y in range(0, max(1, image.height - size), stride):
        share_rows = ((torch.clamp(top_edges + cell_h, max=y + size) - torch.clamp(top_edges, min=y)).clamp(min=0)
                      / cell_h).double()
        for x in range(0, max(1, image.width - size), stride):
            share_columns = ((torch.clamp(left_edges + cell_w, max=x + size) - torch.clamp(left_edges, min=x))
                             .clamp(min=0) / cell_w).double()
            windows.append((float(share_rows @ grid @ share_columns), x, y))
    chosen = []
    for _, x, y in sorted(windows, key=lambda window: -window[0]):
        if all(abs(x - bx) >= size * 0.6 or abs(y - by) >= size * 0.6 for bx, by, _, _ in chosen):
            chosen.append((x, y, x + size, y + size))
        if len(chosen) == count:
            break
    return chosen


def covered(box, centres):
    return {i for i, (y, x) in enumerate(centres) if box[0] <= x < box[2] and box[1] <= y < box[3]}


def fit_picker(kind, maps, width, device, epochs, seed):
    """maps: one (tokens, counts, (rows, columns)) per map. The MLP learns from random batches of tokens; the CNN
    from batches of whole maps of the same size, so that its neighbours are real neighbours."""
    torch.manual_seed(seed)
    picker = PICKERS[kind](width).to(device)
    optimizer = torch.optim.AdamW(picker.parameters(), lr=1e-3, weight_decay=0.01)
    loss_fn = torch.nn.PoissonNLLLoss(log_input=True)
    if kind == "mlp":
        flat_features, flat_labels = torch.cat([m[0] for m in maps]), torch.cat([m[1] for m in maps])
    by_shape = {}
    for i, (_, _, shape) in enumerate(maps):
        by_shape.setdefault(shape, []).append(i)
    for epoch in range(epochs):
        batches = []  # (features, labels, shape) for this epoch
        if kind == "mlp":
            order = torch.randperm(len(flat_labels))
            for start in range(0, len(order), 4096):
                batch = order[start: start + 4096]
                batches.append((flat_features[batch], flat_labels[batch], None))
        else:
            for shape, members in by_shape.items():
                members = [members[i] for i in torch.randperm(len(members)).tolist()]
                for start in range(0, len(members), 8):  # 8 maps is about 3,600 tokens, like the MLP's batch
                    chosen = members[start: start + 8]
                    batches.append((torch.cat([maps[i][0] for i in chosen]), torch.cat([maps[i][1] for i in chosen]),
                                    shape))
            random.Random(seed + epoch).shuffle(batches)
        total, seen = 0.0, 0
        for features, labels, shape in batches:
            loss = loss_fn(picker(features.to(device).float(), shape), labels.to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(labels)
            seen += len(labels)
        print(f"picker {kind} epoch {epoch + 1}: loss {total / seen:.4f}", flush=True)
    picker.eval()
    return picker


def train_picker(model, processor, records, device, amp_dtype, out_dir, kinds=("mlp",), epochs=8, seed=0):
    """Count small regions per token with the rule, train each kind of picker on the same tokens, and check it.

    The last tenth of the maps is held out. On those: the picker's loss (the same Poisson loss it trained
    on), and how many of the small regions inside the rule's first tile are also inside the picker's.
    Returns {kind: picker}; the MLP is saved as picker.pt, the others as picker_<kind>.pt.
    """
    split = max(1, len(records) // 10)
    held_out, training = records[:split], records[split:]
    maps = []
    began = time.time()
    for n, record in enumerate(training, 1):
        image = Image.open(record["image_path"]).convert("RGB")
        tokens, (rows, columns) = map_tokens(model, processor, image, device, amp_dtype)
        maps.append((tokens.half().cpu(), token_counts(image, rows, columns), (rows, columns)))
        if n % 500 == 0:
            print(f"picker: counted {n}/{len(training)} maps ({(time.time() - began) / 60:.1f} min)", flush=True)
    width, tokens_seen = maps[0][0].shape[1], sum(len(m[1]) for m in maps)
    small_regions_counted = int(sum(m[1].sum() for m in maps))
    pickers = {}
    for kind in kinds:
        pickers[kind] = fit_picker(kind, maps, width, device, epochs, seed)
        torch.save(pickers[kind].state_dict(), out_dir / ("picker.pt" if kind == "mlp" else f"picker_{kind}.pt"))
    del maps

    loss_fn = torch.nn.PoissonNLLLoss(log_input=True, reduction="sum")
    kept, wanted, same_place, held_loss, held_tokens = {k: 0 for k in kinds}, {k: 0 for k in kinds}, \
        {k: 0 for k in kinds}, {k: 0.0 for k in kinds}, 0
    for record in held_out:
        image = Image.open(record["image_path"]).convert("RGB")
        tokens, (rows, columns) = map_tokens(model, processor, image, device, amp_dtype)
        labels = token_counts(image, rows, columns)
        held_tokens += len(labels)
        body = map_body(image)
        centres, _ = small_regions(image, body, boundary_mask(image, body))
        rule = busiest_tiles(image, tiles=1)
        for kind, picker in pickers.items():
            with torch.no_grad():
                logs = picker(tokens, (rows, columns))
            held_loss[kind] += loss_fn(logs.cpu(), labels).item()
            if not rule or not len(centres):
                continue
            target = covered(rule[0], centres)
            mine = covered(picker_boxes(logs.exp().cpu(), rows, columns, image, 1)[0], centres)
            kept[kind] += len(target & mine)
            wanted[kind] += len(target)
            same_place[kind] += int(len(target & mine) >= 0.8 * len(target))
    report = {}
    for kind in kinds:
        report[kind] = {"training_maps": len(training), "held_out_maps": len(held_out), "tokens": tokens_seen,
                        "small_regions_counted": small_regions_counted,
                        "held_out_loss": held_loss[kind] / held_tokens,
                        "rule_tile_regions_also_in_picker_tile": kept[kind] / max(1, wanted[kind]),
                        "maps_where_picker_tile_holds_80pct_of_rule_tile": same_place[kind] / len(held_out)}
        print(f"picker {kind}:", json.dumps(report[kind]), flush=True)
    (out_dir / "picker_report.json").write_text(json.dumps(report, indent=1))
    return pickers


def probe_picker(model, processor, picker, sets, device, amp_dtype, out_dir, suffix=""):
    """Does the picker follow small regions, or remember where they usually are?

    Every MapQA map is also shown mirrored and flipped, which moves the north-east cluster to another
    corner. The rule reads only pixels, so it follows the regions wherever they go and is the reference.
    For each version: the share of the map's small regions inside the picker's two tiles against the rule's
    two tiles, and how often the picker's first tile still sits in the top-right quarter, where the
    north-east is on a normally drawn US map. India maps (as drawn) are a geography it never saw.
    """
    flips = {"as drawn": lambda image: image,
             "mirrored left-right": lambda image: image.transpose(Image.FLIP_LEFT_RIGHT),
             "flipped top-bottom": lambda image: image.transpose(Image.FLIP_TOP_BOTTOM),
             "rotated 180": lambda image: image.transpose(Image.ROTATE_180)}
    report = {}
    for name, records in sets:
        conditions = flips if name != "india" else {"as drawn": flips["as drawn"]}
        for label, flip in conditions.items():
            rule_share, picker_share, top_right = [], [], 0
            for record in records:
                image = flip(Image.open(record["image_path"]).convert("RGB"))
                body = map_body(image)
                centres, _ = small_regions(image, body, boundary_mask(image, body))
                if not len(centres):
                    continue
                tokens, (rows, columns) = map_tokens(model, processor, image, device, amp_dtype)
                with torch.no_grad():
                    counts = picker(tokens, (rows, columns)).exp().cpu()
                mine = picker_boxes(counts, rows, columns, image, 2)
                rule = busiest_tiles(image, tiles=2)
                rule_share.append(len(set().union(*[covered(b, centres) for b in rule])) / len(centres))
                picker_share.append(len(set().union(*[covered(b, centres) for b in mine])) / len(centres))
                x = (mine[0][0] + mine[0][2]) / 2 / image.width
                y = (mine[0][1] + mine[0][3]) / 2 / image.height
                top_right += int(x > 0.5 and y < 0.5)
            maps = max(1, len(rule_share))
            report[f"{name}, {label}"] = {
                "maps": len(rule_share),
                "rule_tiles_hold": sum(rule_share) / maps,
                "picker_tiles_hold": sum(picker_share) / maps,
                "picker_first_tile_in_top_right_quarter": top_right / maps}
            print(f"probe {name}, {label}: {json.dumps(report[f'{name}, {label}'])}", flush=True)
    (out_dir / f"picker_probe{suffix}.json").write_text(json.dumps(report, indent=1))


def picker_tiles(model, processor, picker, device, amp_dtype):
    """A box-picking function for tile_images that uses the learned picker instead of the rule."""
    def pick(image, count):
        tokens, (rows, columns) = map_tokens(model, processor, image, device, amp_dtype)
        with torch.no_grad():
            counts = picker(tokens, (rows, columns)).exp().cpu()
        return picker_boxes(counts, rows, columns, image, count)
    return pick


# Boxes in MapQA's 700x500 albers-usa layout; scaled if a map is a different size. NE_BOX was measured
# from the rendered maps: the small north-eastern fills span x 455-549, y 139-258 in every map.
NE_BOX = (440, 138, 565, 263)    # the small north-eastern states: RI, DE, CT, NJ, MA, NH, VT, MD
BODY_BOX = (100, 130, 560, 400)  # the map body, for crops that land elsewhere


def make_crop(image, rng, train):
    """A magnified crop of the map: biased to the north-east in training, fixed there at evaluation.

    Cropping alone would not help - the crop is enlarged so a state that covers a fraction of one
    28x28-pixel vision token in the full map covers several tokens here.
    """
    width, height = image.size
    scale_x, scale_y = width / 700, height / 500
    if train and rng.random() < 0.25:  # elsewhere in the body, so "image 2 zooms image 1" stays general
        left, top, right, bottom = BODY_BOX
        size = rng.uniform(110, 160)
        centre_x = rng.uniform(left + size / 2, right - size / 2)
        centre_y = rng.uniform(top + size / 2, bottom - size / 2)
        zoom = rng.uniform(2.5, 4.0)
    else:
        left, top, right, bottom = NE_BOX
        jitter = rng.uniform(-15, 15) if train else 0.0
        centre_x, centre_y = (left + right) / 2 + jitter, (top + bottom) / 2 + jitter
        size = rng.uniform(110, 165) if train else 130.0
        zoom = rng.uniform(2.5, 4.0) if train else 3.0
    half = size / 2
    box = (max(0, round((centre_x - half) * scale_x)), max(0, round((centre_y - half) * scale_y)),
           min(width, round((centre_x + half) * scale_x)), min(height, round((centre_y + half) * scale_y)))
    crop = image.crop(box)
    return crop.resize((max(28, round(crop.width * zoom)), max(28, round(crop.height * zoom))), Image.BICUBIC)


def tiles_note(boxes):
    """Say where each tile was cut from, so the model can place what it reads back on the map.

    Without this the extra images are just more pixels: the model can see that a region is dark but not
    which region it is, and has to match the tile to the map by eye before it can use it.
    """
    places = [f" view {i} x {left:.2f}-{right:.2f}, y {top:.2f}-{bottom:.2f}."
              for i, (left, top, right, bottom) in enumerate(boxes, start=2)]
    return TILES_NOTE + "".join(places)


def chat_text(processor, target=None, images=1, boxes=None, prompt=PROMPT):
    content = [{"type": "image"} for _ in range(images)]
    if boxes:
        note = tiles_note(boxes)
    else:
        note = "" if images == 1 or prompt == SHEET_PROMPT else (CROP_NOTE if images == 2 else TILES_NOTE)
    content.append({"type": "text", "text": prompt + note})
    messages = [{"role": "user", "content": content}]
    if target is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": target}]})
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=target is None)


class CropDataset(torch.utils.data.Dataset):
    """The full map, and on most examples a magnified crop as a second image.

    A share of examples keep the map alone (--crop-dropout), which keeps single-image inference in
    distribution and lets the crop be ablated at scoring time instead of in a second training run.
    """

    def __init__(self, records, dropout, tiles=0, tile_zoom=3.0, fuse=False):
        self.records, self.dropout = records, dropout
        self.tiles, self.tile_zoom, self.fuse = tiles, tile_zoom, fuse

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        record = self.records[i]
        image = Image.open(record["image_path"]).convert("RGB")
        boxes, fuse = [], False
        if random.random() < self.dropout:
            images = [image]                       # the map alone, so single-image use stays in distribution
        elif self.tiles and self.fuse:
            images, fuse = [image], True           # the tiles are added into the map's own tokens
        elif self.tiles:
            views, boxes = tile_images(image, self.tiles, self.tile_zoom, random, jitter=8)
            images = [image] + views
        else:
            images = [image, make_crop(image, random, train=True)]
        return images, boxes, record["target"], fuse, prompt_for(record)


class TrainCollate:
    """Image(s) + prompt + answer, with labels only on the answer tokens."""

    def __init__(self, processor, fine_processor=None, scale=2, fusion=None):
        self.processor = processor
        self.fusion = fusion  # (tiles, cells, zoom) when tiles are fused into the map's tokens
        self.fine_processor = fine_processor  # same processor, allowed more pixels, for the second scale
        self.scale = scale
        self.answer_start = processor.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

    def __call__(self, batch):
        texts, images = [], []
        fused = []
        for index, (sample_images, boxes, target, fuse, prompt) in enumerate(batch):
            if fuse:
                fused.append((index, sample_images[0]))
            images.extend(sample_images)
            texts.append(chat_text(self.processor, target, len(sample_images), boxes, prompt))
        fine = None
        if self.fine_processor is not None:
            images, fine_images = two_scale_images(self.processor, images, self.scale)
            fine = self.fine_processor.image_processor(images=fine_images, return_tensors="pt")
        enc = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        if fine is not None:
            enc["s2_pixel_values"], enc["s2_grid_thw"] = fine["pixel_values"], fine["image_grid_thw"]
        if fused:
            for key, value in fusion_inputs(self.processor, fused, *self.fusion).items():
                enc[key] = value
        labels = enc["input_ids"].clone()
        labels[enc["attention_mask"] == 0] = -100
        marker = torch.tensor(self.answer_start)
        for row, ids in enumerate(enc["input_ids"]):
            hits = (ids.unfold(0, len(marker), 1) == marker).all(dim=1).nonzero()
            labels[row, : int(hits[-1]) + len(marker)] = -100
        enc["labels"] = labels
        return enc


def predict(model, processor, records, args, device, amp_dtype, crop=False, s2=None, fine_processor=None,
            tiles=0, fusion=None, pick=None, sheets=None):
    def generate(batch):
        """Predictions for one batch; on running out of GPU memory, the batch is split in two and retried,
        so a long run is not lost to one oversized batch at the very end."""
        try:
            return generate_batch(batch)
        except torch.cuda.OutOfMemoryError:
            if len(batch) == 1:
                raise
            torch.cuda.empty_cache()
            half = len(batch) // 2
            print(f"out of memory on a batch of {len(batch)}; retrying in halves", flush=True)
            return generate(batch[:half]) + generate(batch[half:])

    def generate_batch(batch):
        texts, images, fused = [], [], []
        for record in batch:
            image = Image.open(record["image_path"]).convert("RGB")
            images.append(image)
            extra, boxes = [], []
            if tiles and fusion is not None:
                fused.append((len(texts), image))
            elif tiles:
                chosen = pick(image, tiles) if pick else None
                extra, boxes = tile_images(image, tiles, args.tile_zoom, boxes=chosen)
            elif crop:
                extra = [make_crop(image, random, train=False)]
            elif sheets:
                extra = [sheets.scored(image, record)]
            images.extend(extra)
            texts.append(chat_text(processor, images=1 + len(extra), boxes=boxes,
                                   prompt=SHEET_PROMPT if sheets else prompt_for(record)))
        fine = None
        if fine_processor is not None:
            images, fine_images = two_scale_images(processor, images, args.s2_scale)
            fine = fine_processor.image_processor(images=fine_images, return_tensors="pt")
        enc = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)
        if fused:
            seen = fusion_inputs(processor, fused, tiles, args.fusion_cells, args.fusion_zoom)
            fusion.pending = (seen["fx_pixel_values"].to(device), seen["fx_grid_thw"].to(device),
                              seen["fx_where"].to(device))
        if fine is not None:
            s2.pending = (fine["pixel_values"].to(device), fine["image_grid_thw"].to(device))
        with autocast(device, amp_dtype):
            ids = model.generate(**enc, max_new_tokens=args.max_new_tokens, do_sample=False,
                                 repetition_penalty=1.0, use_cache=True)
        return processor.batch_decode(ids[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

    predictions = []
    began = time.time()
    with torch.no_grad():
        for start in range(0, len(records), args.eval_batch_size):
            predictions += generate(records[start : start + args.eval_batch_size])
            print(f"predicted {len(predictions)}/{len(records)} ({(time.time() - began) / 60:.1f} min)", flush=True)
    return predictions


def report_lora(model):
    """Print where LoRA actually landed. PEFT matches module names, so a target that matches nothing is
    silently a no-op; this run's whole question is whether vision attention helps, so it must not pass quietly."""
    names = [name for name, _ in model.named_modules() if name.endswith("lora_A")]
    vision_attention = sum(1 for n in names if ".visual." in n and ".attn." in n)
    vision_mlp = sum(1 for n in names if ".visual." in n and ".mlp." in n and "merger" not in n)
    merger = sum(1 for n in names if "merger" in n)
    print(f"LoRA modules: {len(names)} total | vision attention {vision_attention} | "
          f"vision MLP {vision_mlp} | merger {merger} | language {len(names) - vision_attention - vision_mlp - merger}",
          flush=True)
    return vision_attention


def load_prior_adapter(model, path):
    """Load an earlier adapter into a model whose LoRA now covers more modules.

    LoRA initialises its B matrix to zero, so modules the earlier run did not train start as exact
    no-ops and the model begins exactly where that run ended.
    """
    from safetensors.torch import load_file

    state = load_file(str(Path(path) / "adapter_model.safetensors"))
    result = set_peft_model_state_dict(model, state)
    unexpected = list(getattr(result, "unexpected_keys", []))
    if unexpected:
        raise SystemExit(f"prior adapter does not line up: {len(unexpected)} unexpected keys, "
                         f"e.g. {unexpected[:3]}")
    print(f"loaded {len(state)} tensors from the prior adapter", flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    add_common_args(parser)
    parser.add_argument("--model", default="Qwen/Qwen2.5-VL-3B-Instruct")
    parser.add_argument("--adapter", help="a saved LoRA adapter to continue from or to score")
    parser.add_argument("--lora-r", type=int, default=16)
    parser.add_argument("--max-pixels", type=int, default=640_000,
                        help="larger images are scaled down to this many pixels (MapQA maps have 350,000)")
    parser.add_argument("--vision-lora", action="store_true",
                        help="also train the vision encoder's attention and patch merger, not just its MLPs")
    parser.add_argument("--crop", action="store_true",
                        help="pass a magnified crop of the map as a second image")
    parser.add_argument("--crop-dropout", type=float, default=0.2,
                        help="share of training examples that keep the full map alone")
    parser.add_argument("--eval-both", action="store_true",
                        help="with --crop, also score without the crop, to separate its effect")
    parser.add_argument("--tiles", type=int, default=0,
                        help="magnify this many of the map's most crowded areas and pass them as extra "
                             "images; the areas are found from the picture, not from known geography")
    parser.add_argument("--tile-zoom", type=float, default=3.0, help="how much to magnify each tile")
    parser.add_argument("--fusion", action="store_true",
                        help="with --tiles: add each tile's close-up tokens into the map's own tokens instead of "
                             "passing the tile as an extra image, so the token count does not grow")
    parser.add_argument("--fusion-lr", type=float,
                        help="learning rate for the fusion weights alone (default: --lr); they start from nothing")
    parser.add_argument("--fusion-init-std", type=float, default=0.0,
                        help="start the fusion output layer at this small spread instead of exactly zero")
    parser.add_argument("--sheet", type=Path, help="a folder of region labels: add a region sheet as a second image "
                                                   "(train/sheet.py); synthetic maps only, so MapQA is not scored")
    parser.add_argument("--picker", choices=["rule", "mlp", "cnn", "both"], default="rule",
                        help="mlp / cnn / both: train small networks on the vision tokens to pick the tiles, and "
                             "score with the rule's tiles and each network's, on the same weights")
    parser.add_argument("--skip-rule-eval", action="store_true",
                        help="with --picker: score only the learned pickers' tiles, not the rule's")
    parser.add_argument("--probe-trained", action="store_true",
                        help="with --picker: after training, probe each new picker on mirrored / flipped maps and "
                             "India (as --probe-picker does), writing picker_probe_<kind>.json")
    parser.add_argument("--picker-maps", type=int, default=2000, help="training maps used to teach the picker")
    parser.add_argument("--probe-picker", type=Path,
                        help="a folder holding a trained picker.pt: check where it puts tiles on mirrored and "
                             "flipped maps (and India maps with --india), then stop; nothing is generated")
    parser.add_argument("--fusion-cells", type=int, default=3,
                        help="a fused tile covers this many coarse tokens on each side")
    parser.add_argument("--fusion-zoom", type=int, default=3,
                        help="whole-number magnification of a fused tile (fine tokens per coarse token, per side)")
    parser.add_argument("--s2", action="store_true",
                        help="run the vision tower at two scales and fuse them, instead of cropping by hand")
    parser.add_argument("--s2-scale", type=int, default=2,
                        help="how many times larger the second pass sees the map")
    # A T4 trains about 2.3 s per MapQA map; 8 h leaves room for scoring inside Kaggle's 12-hour limit
    parser.set_defaults(epochs=1, hours=8.0, lr=2e-4, batch_size=2, grad_accum=8, eval_batch_size=16,
                        max_new_tokens=600)
    args = parser.parse_args()
    set_seed(args.seed)
    args.out.mkdir(parents=True, exist_ok=True)
    (args.out / "args.json").write_text(json.dumps(vars(args), default=str, indent=1))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":  # T4 and P100 have no bfloat16
        dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    else:
        dtype = torch.float32
    amp_dtype = dtype if device.type == "cuda" else None
    root = find_mapqa_root(args.mapqa)
    processor = AutoProcessor.from_pretrained(args.model, max_pixels=args.max_pixels)
    processor.tokenizer.padding_side = "left"
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(args.model, torch_dtype=dtype,
                                                               attn_implementation="sdpa").to(device)
    targets = VISION_LORA_REGEX if args.vision_lora else LORA_TARGETS
    lora = LoraConfig(r=args.lora_r, lora_alpha=2 * args.lora_r, lora_dropout=0.05,
                      target_modules=targets, task_type="CAUSAL_LM")
    if args.adapter and args.vision_lora:
        # the saved adapter covers fewer modules than this config, so build the wider one and fill it in
        model = get_peft_model(model, lora)
        load_prior_adapter(model, find_saved(args.adapter, "adapter_config.json"))
    elif args.adapter:
        model = PeftModel.from_pretrained(model, find_saved(args.adapter, "adapter_config.json"),
                                          is_trainable=not args.skip_train)
    elif not args.skip_train:
        model = get_peft_model(model, lora)
    if args.adapter or not args.skip_train:
        vision_attention = report_lora(model)
        if args.vision_lora and not vision_attention:
            raise SystemExit("--vision-lora matched no vision attention; check VISION_LORA_REGEX")

    if args.fusion and (args.s2 or not args.tiles):
        raise SystemExit("--fusion needs --tiles N and cannot be combined with --s2")
    fusion = None
    if args.fusion:
        fusion = attach_fusion(model, args.fusion_cells, args.fusion_zoom, device, args.fusion_init_std)
        added = sum(p.numel() for p in (*fusion.reduce.parameters(), *fusion.out.parameters()))
        print(f"fusion: {args.tiles} tiles of {args.fusion_cells}x{args.fusion_cells} coarse tokens at "
              f"{args.fusion_zoom}x, {added:,} trainable weights", flush=True)

    s2 = fine_processor = None
    if args.s2:
        s2 = attach_s2(model, args.s2_scale, device)
        fine_processor = AutoProcessor.from_pretrained(args.model,
                                                       max_pixels=args.max_pixels * args.s2_scale ** 2)
        print(f"S2: second pass at {args.s2_scale}x the size, fusing into "
              f"{s2.fuse.weight.numel():,} trainable weights", flush=True)

    sheets = Sheets(args.sheet) if args.sheet else None
    if not args.skip_train:
        model.print_trainable_parameters()
        model.config.use_cache = False
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
        train = training_records(args, root)
        dataset = SheetDataset(train, sheets) if sheets else CropDataset(
            train, args.crop_dropout if (args.crop or args.tiles) else 1.0, args.tiles, args.tile_zoom, fuse=args.fusion)
        collate = TrainCollate(processor, fine_processor, args.s2_scale,
                               (args.tiles, args.fusion_cells, args.fusion_zoom) if args.fusion else None)
        loader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True, num_workers=args.workers,
                            collate_fn=collate)

        def forward(batch):
            batch = batch.to(device)
            if s2 is not None:
                s2.pending = (batch.pop("s2_pixel_values"), batch.pop("s2_grid_thw"))
            if fusion is not None:  # absent when every example in the batch kept the map alone
                pixels = batch.pop("fx_pixel_values", None)
                grid, where = batch.pop("fx_grid_thw", None), batch.pop("fx_where", None)
                fusion.pending = None if pixels is None else (pixels, grid, where)
            return model(**batch).loss

        groups = None
        if fusion is not None and args.fusion_lr:
            own = [*fusion.reduce.parameters(), *fusion.out.parameters()]
            ids = {id(p) for p in own}
            rest = [p for p in model.parameters() if p.requires_grad and id(p) not in ids]
            groups = [{"params": rest}, {"params": own, "lr": args.fusion_lr}]
            print(f"fusion weights learn at {args.fusion_lr}, the rest at {args.lr}", flush=True)
        print(f"{len(loader)} batches per epoch", flush=True)
        train_loop(model, loader, args, forward, args.out, amp_dtype, groups)
        model.save_pretrained(args.out / "adapter")
        if s2 is not None:  # the fusion is not part of the LoRA adapter, so it is saved beside it
            torch.save(s2.fuse.state_dict(), args.out / "s2_fuse.pt")
        if fusion is not None:
            torch.save({"reduce": fusion.reduce.state_dict(), "out": fusion.out.state_dict()},
                       args.out / "fusion.pt")

    model.eval()
    if device.type == "cuda":
        torch.cuda.empty_cache()  # release memory held from training before generating
    if args.probe_picker:
        picker = TokenPicker(vision_tower(model).config.out_hidden_size).to(device)
        weights = Path(find_saved(args.probe_picker, "picker.pt")) / "picker.pt"
        picker.load_state_dict(torch.load(weights, map_location=device))
        picker.eval()
        probe_picker(model, processor, picker, evaluation_sets(args, root), device, amp_dtype, args.out)
        return

    variants = [] if args.skip_rule_eval and args.picker != "rule" else [("", args.crop or bool(args.tiles), None)]
    if args.picker != "rule":
        if not args.tiles or fusion is not None:
            raise SystemExit("--picker picks tiles for --tiles N, passed as images")
        kinds = ("mlp", "cnn") if args.picker == "both" else (args.picker,)
        teach = sample_records(training_records_mapqa(args, root), args.picker_maps, seed=args.seed + 1)
        pickers = train_picker(model, processor, teach, device, amp_dtype, args.out, kinds=kinds)
        if args.probe_trained:
            for kind, picker in pickers.items():
                print(f"probing the {kind} picker", flush=True)
                probe_sets = [(name, records[:200]) for name, records in evaluation_sets(args, root)]
                probe_picker(model, processor, picker, probe_sets, device, amp_dtype, args.out, suffix=f"_{kind}")
        for kind, picker in pickers.items():
            variants.append((f"_{kind}", True, picker_tiles(model, processor, picker, device, amp_dtype)))
    if (args.crop or args.tiles) and args.eval_both:
        variants.append(("_nocrop", False, None))  # same weights, extra views withheld: their effect on its own
    for name, records in evaluation_sets(args, root):
        for suffix, use_crop, pick in variants:
            print(f"scoring {len(records)} {name} test maps{' without the crop' if suffix == '_nocrop' else ''}"
                  f"{' with the learned picker' if pick else ''}", flush=True)
            predictions = predict(model, processor, records, args, device, amp_dtype, crop=use_crop,
                                  s2=s2, fine_processor=fine_processor,
                                  tiles=args.tiles if use_crop else 0, fusion=fusion, pick=pick, sheets=sheets)
            evaluate_and_save(records, predictions, args.out, name + suffix)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile answer_sheet.py
#!/usr/bin/env python3
"""Learned answer sheet: one model that finds a map's regions itself and reads each one, with no fitting or OCR.

The region sheet showed the model reads a region well once that region is handed to it by name. Here the model
makes that hand-over itself:

1. Region head. A small head on the vision tower's 14-px patches (before Qwen merges them into 28-px tokens)
   says which region each patch belongs to. Its vocabulary holds every region of every country it is taught
   (US and India here), so it also says which country the map shows.
2. Region tokens. For each region, the patches the head gives it are averaged and passed through Qwen's own
   patch merger, giving one token that looks like a token filled with that region. No new projection is learned.
3. The language model sees the map, then "code: [region token]" for every region of the country, and writes the
   usual JSON, reading the legend from the map itself.

Shapefiles appear only in teaching: they label which region lies under each patch (synthetic maps from the
generator; MapQA and real maps from boundary fits made beforehand by lookup/answer_labels.py).

Stage A teaches the head alone on frozen tokens. Stage B trains LoRA and the head together on the JSON answer,
the head keeping its patch labels as a side loss. In training, half the maps show a random set of regions under
random codes, so the answer has to come from each region token, not from memorised geography.

Scored on: the head's patches (synthetic India, MapQA test, real India), real India maps (predictions only; scored
locally by lookup/score_real.py), 300 synthetic India test maps and the 400 MapQA test maps.

    python kaggle/launch.py answer --tag v1 --india-dataset sartma/synth-india-choropleth \
        --extra-dataset sartma/region-probe --extra-dataset sartma/answer-labels --extra-dataset sartma/real-sheets \
        --from-run sartma/mapqa-derender-qwen --no-wait -- --adapter /kaggle/input
"""

import argparse
import json
import random
import string
import time
from pathlib import Path

import cv2
import numpy as np
import torch
from peft import PeftModel
from PIL import Image
from torch.utils.data import DataLoader
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

from common import (add_common_args, autocast, evaluate_and_save, find_images_root, find_saved, read_split,
                    sample_records, set_seed, train_loop)
from train_qwen import report_lora

PROMPT = (
    "The first image is a choropleth map. After this text, each region of the map is listed as its code followed by "
    "one image token showing that region. Return only JSON with these keys: "
    '"title"; "legend_title" (null if there is none); "legend_type" ("continuous" or "discrete"); '
    '"data_type" ("percentage" or "number"); "ranges" (discrete legends only: the legend ranges, lowest first); '
    '"values" (every listed region by its code, in the order listed: a number for a continuous legend, the index of '
    "the region's range for a discrete legend, null for no data). Regions:"
)
SAMPLES = 4  # label points per patch side
SMALL = {"US": {"RI", "DE", "DC", "CT", "NJ", "VT", "NH", "MA", "MD", "HI"}, "IN": {"AN", "DL", "GA", "SK", "TR"}}


# ---------------------------------------------------------------- labels

class Labels:
    """Which region lies under each pixel, as classes of one vocabulary shared by every country."""

    def __init__(self, root):
        root = Path(root)
        self.vocab = json.loads(next(root.rglob("vocab.json")).read_text())
        self.folder = next(root.rglob("vocab.json")).parent
        self.index = {name: i for i, name in enumerate(self.vocab)}
        probe = next(root.rglob("regions_usa.json")).parent
        self.synth = {}
        for prefix, name, country in (("synthind_", "regions", "IN"), ("synth_", "regions_usa", "US")):
            meta = json.loads((probe / f"{name}.json").read_text())
            raster = np.asarray(Image.open(probe / f"{name}.png"))
            lut = np.zeros(256, np.uint8)
            for k, code in enumerate(meta["codes"], 1):
                lut[k] = self.index[f"{country}-{code}"]
            self.synth[prefix] = (meta, lut[raster])
        self.real = probe / "real"
        india = json.loads((probe / "regions.json").read_text())
        usa = json.loads((probe / "regions_usa.json").read_text())
        self.listed = {"IN": india["listed"], "US": usa["codes"]}

    @staticmethod
    def country(record):
        return "US" if record["id"].startswith(("synth_", "map_")) else "IN"

    def label(self, record, size):
        """(height, width) uint8 class numbers for this record's image, or None when it has no labels."""
        name = record["id"]
        for prefix, (meta, raster) in self.synth.items():
            if name.startswith(prefix):
                left, top, right, bottom = meta["placed"][name]
                place = np.array([[(right - left) / raster.shape[1], 0, left], [0, (bottom - top) / raster.shape[0], top]])
                return cv2.warpAffine(raster, place, size, flags=cv2.INTER_NEAREST)
        for folder in ("mapqa", "real"):
            path = self.folder / folder / f"{name}.png"
            if path.exists():
                label = np.asarray(Image.open(path))
                return label if label.shape[::-1] == tuple(size) else cv2.resize(label, size, interpolation=cv2.INTER_NEAREST)
        return None


def move(image, label, rng):
    """Map and labels shrunk, stretched a little and placed at random on a page of the map's own colour."""
    scale_x = rng.uniform(0.6, 1.0)
    scale_y = min(1.0, scale_x * rng.uniform(0.9, 1.1))
    width, height = max(1, int(image.width * scale_x)), max(1, int(image.height * scale_y))
    x, y = rng.randint(0, image.width - width), rng.randint(0, image.height - height)
    page = Image.new("RGB", image.size, image.getpixel((1, 1)))
    page.paste(image.resize((width, height), Image.BILINEAR), (x, y))
    moved = None
    if label is not None:
        moved = np.zeros_like(label)
        moved[y: y + height, x: x + width] = cv2.resize(label, (width, height), interpolation=cv2.INTER_NEAREST)
    return page, moved


def patch_shares(label, rows, columns, classes):
    """(rows * columns, classes): the share of each 14-px patch that each class covers, patches row by row."""
    height, width = label.shape
    xs = np.minimum(((np.arange(columns * SAMPLES) + 0.5) / (columns * SAMPLES) * width).astype(int), width - 1)
    ys = np.minimum(((np.arange(rows * SAMPLES) + 0.5) / (rows * SAMPLES) * height).astype(int), height - 1)
    found = label[np.ix_(ys, xs)].astype(np.int64)
    patch = (np.arange(len(ys)) // SAMPLES)[:, None] * columns + (np.arange(len(xs)) // SAMPLES)[None, :]
    counts = np.bincount((patch * classes + found).ravel(), minlength=rows * columns * classes)
    return torch.from_numpy(counts.reshape(-1, classes) / SAMPLES ** 2).float()


# ---------------------------------------------------------------- the model's new parts

class RegionHead(torch.nn.Module):
    """Patch features (rows, columns, width) -> class logits (rows * columns, classes), seeing a few neighbours."""

    def __init__(self, width, classes, hidden=256):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Conv2d(width, hidden, 1), torch.nn.GELU(),
            torch.nn.Conv2d(hidden, hidden, 3, padding=1), torch.nn.GELU(),
            torch.nn.Conv2d(hidden, hidden, 3, padding=2, dilation=2), torch.nn.GELU(),
            torch.nn.Conv2d(hidden, hidden, 3, padding=4, dilation=4), torch.nn.GELU(),
            torch.nn.Conv2d(hidden, classes, 1))

    def forward(self, grid):
        out = self.net(grid.permute(2, 0, 1)[None].float())[0]
        return out.permute(1, 2, 0).flatten(0, 1)


class AnswerSheetVisual(torch.nn.Module):
    """The vision tower, plus region tokens for the 28-px placeholder images that follow each map.

    Images arrive as: map, its placeholders, next map, its placeholders... A placeholder is any 2x2-patch image.
    `pending` gives, per map, the class of each of its placeholders and (in training) the map's patch labels.
    """

    def __init__(self, visual, classes, sharpen=2.0):
        super().__init__()
        self.visual = visual
        self.head = RegionHead(visual.config.hidden_size, classes)
        self.sharpen = sharpen
        self.pending, self.aux_loss, self._captured = None, None, None
        visual.merger.register_forward_pre_hook(self._capture)

    def _capture(self, module, args):
        self._captured = args[0]

    def __getattr__(self, name):  # dtype, spatial_merge_size, config and friends come from the tower
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self._modules["visual"], name)

    def patch_grids(self, hidden_states, grid_thw):
        """Run the tower on maps only: (merged tokens, [(rows, columns, width) patch features per map])."""
        tokens = self.visual(hidden_states, grid_thw=grid_thw)
        before = self._captured
        window_index, _ = self.visual.get_window_index(grid_thw)
        before = before.reshape(-1, 4, before.shape[-1])[torch.argsort(window_index)]
        grids, start = [], 0
        for _, height, width in grid_thw.tolist():
            count = height * width // 4
            part = before[start: start + count].reshape(height // 2, width // 2, 2, 2, -1)
            grids.append(part.permute(0, 2, 1, 3, 4).reshape(height, width, -1))
            start += count
        return tokens, grids

    def forward(self, hidden_states, grid_thw, **kwargs):
        pending, self.pending = self.pending, None
        placeholder = (grid_thw[:, 1] == 2) & (grid_thw[:, 2] == 2)
        if pending is None or not placeholder.any():
            return self.visual(hidden_states, grid_thw=grid_thw, **kwargs)
        chunks = torch.split(hidden_states, grid_thw.prod(-1).tolist())
        maps = [i for i in range(len(chunks)) if not placeholder[i]]
        tokens, grids = self.patch_grids(torch.cat([chunks[i] for i in maps]), grid_thw[maps])
        sizes = [h * w // 4 for _, h, w in grid_thw[maps].tolist()]
        out, losses = [], []
        for m, (map_tokens, grid) in enumerate(zip(torch.split(tokens, sizes), grids)):
            logits = self.head(grid)
            shares = pending["labels"][m]
            if shares is not None:
                losses.append(-(shares.to(logits.device) * logits.float().log_softmax(-1)).sum(-1).mean())
            regions = self.region_tokens(grid, logits, pending["classes"][m]).to(map_tokens.dtype)
            for j, token in pending.get("override", {}).get(m, {}).items():  # e.g. made from a zoomed view
                regions[j] = token.to(regions.dtype)
            out += [map_tokens, regions]
        self.aux_loss = torch.stack(losses).mean() if losses else None
        return torch.cat(out)

    def region_tokens(self, grid, logits, classes):
        """One token per class: its patches averaged, weighted by the head's (sharpened) belief, then merged."""
        chances = logits.float().softmax(-1).detach()[:, classes] ** self.sharpen
        return self.pool(grid, chances)

    def pool(self, grid, weights):
        """grid (rows, columns, width), weights (rows * columns, k) -> k tokens through Qwen's own merger."""
        weights = weights / weights.sum(0, keepdim=True).clamp_min(1e-6)
        flat = grid.reshape(-1, grid.shape[-1])
        pooled = weights.T.to(flat.dtype) @ flat
        return self.visual.merger(pooled.repeat_interleave(4, dim=0))


def attach(model, classes, device):
    for module in model.modules():
        if "visual" in module._modules and not isinstance(module._modules["visual"], AnswerSheetVisual):
            wrapper = AnswerSheetVisual(module._modules["visual"], classes)
            wrapper.head.to(device=device, dtype=torch.float32)
            module._modules["visual"] = wrapper
            return wrapper
    raise SystemExit("could not find the vision tower")


WRAPPED, PLAIN = ".visual.visual.", ".visual."


def load_adapter(model, folder, trainable):
    """PeftModel.from_pretrained that also reads adapters saved with the wrapper attached (v1's): their vision LoRA
    keys say visual.visual.*, which peft cannot match on a plain model and silently leaves at zero."""
    import shutil
    import tempfile
    from safetensors.torch import load_file, save_file
    folder = Path(folder)
    weights = load_file(folder / "adapter_model.safetensors")
    if any(WRAPPED in k for k in weights):
        fixed = Path(tempfile.mkdtemp())
        for f in folder.iterdir():
            if f.is_file() and f.name != "adapter_model.safetensors":
                shutil.copy(f, fixed / f.name)
        save_file({k.replace(WRAPPED, PLAIN): v for k, v in weights.items()}, fixed / "adapter_model.safetensors")
        print(f"renamed {sum(WRAPPED in k for k in weights)} wrapped vision keys", flush=True)
        folder = fixed
    model = PeftModel.from_pretrained(model, folder, is_trainable=trainable)
    if any("visual" in k for k in weights):  # a fresh LoRA's B is zero, so a zero B means its weights never loaded
        empty = [n for n, p in model.named_parameters() if "visual" in n and "lora_B" in n and not p.detach().abs().sum()]
        if empty:
            raise SystemExit(f"{len(empty)} vision LoRA weights did not load, e.g. {empty[0]}")
    return model


def save_adapter(model, folder):
    """save_pretrained, with the vision keys written as on a plain model so the adapter loads anywhere."""
    from safetensors.torch import load_file, save_file
    model.save_pretrained(folder)
    path = Path(folder) / "adapter_model.safetensors"
    weights = load_file(path)
    save_file({k.replace(WRAPPED, PLAIN): v for k, v in weights.items()}, path)


# ---------------------------------------------------------------- inputs

PLACEHOLDER = Image.new("RGB", (28, 28), (128, 128, 128))


def chat(processor, codes, target=None):
    content = [{"type": "image"}, {"type": "text", "text": PROMPT}]
    for code in codes:
        content += [{"type": "text", "text": f" {code}:"}, {"type": "image"}]
    messages = [{"role": "user", "content": content}]
    if target is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": target}]})
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=target is None)


class TrainSet(torch.utils.data.Dataset):
    def __init__(self, records, labels, move_share=0.5, rename_share=0.5):
        self.records, self.labels = records, labels
        self.move_share, self.rename_share = move_share, rename_share

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        rng = random.Random()
        record = self.records[i]
        image = Image.open(record["image_path"]).convert("RGB")
        label = self.labels.label(record, image.size)
        if rng.random() < self.move_share:
            image, label = move(image, label, rng)
        target = json.loads(record["target"])
        country = Labels.country(record)
        real = list(target["values"])
        if rng.random() < self.rename_share:  # a random set of regions under random codes: copy, don't recall
            chosen = rng.sample(real, rng.randint(min(16, len(real)), len(real)))
            names = rng.sample([a + b for a in string.ascii_uppercase for b in string.ascii_uppercase], len(chosen))
            shown = sorted(zip(names, chosen))
        else:
            shown = [(c, c) for c in real]
        target["values"] = {name: target["values"][code] for name, code in shown}
        classes = [self.labels.index[f"{country}-{code}"] for _, code in shown]
        return image, label, [name for name, _ in shown], classes, json.dumps(target, ensure_ascii=False,
                                                                              separators=(",", ":"))


class Collate:
    def __init__(self, processor, classes):
        self.processor, self.classes = processor, classes
        self.answer_start = processor.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

    def __call__(self, batch):
        texts, images, labels, classes = [], [], [], []
        for image, label, names, region_classes, target in batch:
            texts.append(chat(self.processor, names, target))
            images += [image] + [PLACEHOLDER] * len(names)
            labels.append(label)
            classes.append(region_classes)
        enc = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        grids = [g for g in enc["image_grid_thw"].tolist() if g[1:] != [2, 2]]
        shares = [None if label is None else patch_shares(label, h, w, self.classes)
                  for label, (_, h, w) in zip(labels, grids)]
        ids = enc["input_ids"]
        answer = ids.clone()
        answer[enc["attention_mask"] == 0] = -100
        marker = torch.tensor(self.answer_start)
        for row in range(len(ids)):
            hits = (ids[row].unfold(0, len(marker), 1) == marker).all(dim=1).nonzero()
            answer[row, : int(hits[-1]) + len(marker)] = -100
        enc["labels"] = answer
        return enc, {"classes": classes, "labels": shares}


# ---------------------------------------------------------------- stage A: the head alone

def map_grid(wrapper, processor, image, device, amp_dtype):
    seen = processor.image_processor(images=[image], return_tensors="pt")
    with torch.no_grad(), autocast(device, amp_dtype):
        _, grids = wrapper.patch_grids(seen["pixel_values"].to(device).type(wrapper.visual.dtype),
                                       seen["image_grid_thw"].to(device))
    return grids[0]


def teach_head(wrapper, processor, records, labels, device, amp_dtype, minutes, seed):
    """The head alone on frozen patch features, one map at a time, half of them moved."""
    head = wrapper.head
    optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=0.01)
    deadline, began, step, window = time.time() + minutes * 60, time.time(), 0, []
    rng = random.Random(seed)
    head.train()
    while time.time() < deadline:
        for record in rng.sample(records, len(records)):
            image = Image.open(record["image_path"]).convert("RGB")
            label = labels.label(record, image.size)
            if rng.random() < 0.5:
                image, label = move(image, label, rng)
            grid = map_grid(wrapper, processor, image, device, amp_dtype)
            shares = patch_shares(label, grid.shape[0], grid.shape[1], len(labels.vocab)).to(device)
            loss = -(shares * head(grid).log_softmax(-1)).sum(-1).mean()
            (loss / 4).backward()
            window.append(loss.item())
            step += 1
            if step % 4 == 0:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
            if step % 200 == 0:
                print(json.dumps({"head_step": step, "loss": round(float(np.mean(window)), 4),
                                  "minutes": round((time.time() - began) / 60, 1)}), flush=True)
                window.clear()
            if time.time() > deadline:
                break
    head.eval()
    print(f"head taught on {step} maps", flush=True)


# ---------------------------------------------------------------- scoring

def country_of(chances, vocab):
    """The country whose regions hold most of the head's belief over the page."""
    mass = chances[:, 1:].sum(0)
    totals = {}
    for k, name in enumerate(vocab[1:]):
        totals[name[:2]] = totals.get(name[:2], 0.0) + float(mass[k])
    return max(totals, key=totals.get)


def head_report(wrapper, processor, sets, labels, device, amp_dtype):
    """Per set: country right, patch accuracy, and how pure each region's token is (the share of its pooling
    weight that falls on that region's own pixels), big and small regions apart."""
    report = {}
    for name, records, moved in sets:
        rng = random.Random(1)
        right_country, patch_right, patches = 0, 0, 0
        purity = {"big": [], "small": []}
        for record in records:
            image = Image.open(record["image_path"]).convert("RGB")
            label = labels.label(record, image.size)
            if label is None:
                continue
            if moved:
                image, label = move(image, label, rng)
            grid = map_grid(wrapper, processor, image, device, amp_dtype)
            with torch.no_grad():
                chances = wrapper.head(grid).float().softmax(-1).cpu()
            shares = patch_shares(label, grid.shape[0], grid.shape[1], len(labels.vocab))
            country = Labels.country(record)
            right_country += country_of(chances, labels.vocab) == country
            on_map = shares[:, 1:].sum(1) > 0.5
            patch_right += int((chances[on_map].argmax(1) == shares[on_map].argmax(1)).sum())
            patches += int(on_map.sum())
            for code in labels.listed[country]:
                k = labels.index[f"{country}-{code}"]
                if shares[:, k].sum() == 0:
                    continue
                weights = chances[:, k] ** wrapper.sharpen
                weights = weights / weights.sum().clamp_min(1e-6)
                purity["small" if code in SMALL[country] else "big"].append(float((weights * shares[:, k]).sum()))
        report[name] = {"maps": len(records), "country_right": right_country / max(1, len(records)),
                        "patch_accuracy": patch_right / max(1, patches),
                        **{f"{size}_purity_median": float(np.median(v)) if v else None for size, v in purity.items()},
                        **{f"{size}_purity_over_0.8": float(np.mean(np.asarray(v) > 0.8)) if v else None
                           for size, v in purity.items()}}
        print(name, json.dumps(report[name]), flush=True)
    return report


def predict(model, wrapper, processor, records, labels, args, device, amp_dtype, force=None):
    """Each map: the head names the country (or `force` does), then the model reads every listed region of it."""
    answers, countries, began = [], [], time.time()
    for start in range(0, len(records), args.eval_batch_size):
        batch = records[start: start + args.eval_batch_size]
        texts, images, classes = [], [], []
        for record in batch:
            image = Image.open(record["image_path"]).convert("RGB")
            grid = map_grid(wrapper, processor, image, device, amp_dtype)
            with torch.no_grad():
                country = country_of(wrapper.head(grid).float().softmax(-1).cpu(), labels.vocab)
            country = force or country
            codes = labels.listed[country]
            countries.append(country)
            texts.append(chat(processor, codes))
            images += [image] + [PLACEHOLDER] * len(codes)
            classes.append([labels.index[f"{country}-{c}"] for c in codes])
        enc = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)
        wrapper.pending = {"classes": classes, "labels": [None] * len(batch)}
        with torch.no_grad(), autocast(device, amp_dtype):
            ids = model.generate(**enc, max_new_tokens=args.max_new_tokens, do_sample=False, use_cache=True)
        answers += processor.batch_decode(ids[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"predicted {len(answers)}/{len(records)} ({(time.time() - began) / 60:.1f} min)", flush=True)
    return answers, countries


def real_records(root):
    """The linked real India maps (records.jsonl beside maps/), with no targets: scored locally."""
    path = next(Path(root).rglob("records.jsonl"), None)
    if path is None:
        return []
    return [{"id": r["id"], "image_path": str(path.parent / r["map"])} for r in map(json.loads, open(path))]


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    add_common_args(parser)
    parser.add_argument("--labels", type=Path, default=Path("/kaggle/input"),
                        help="holds vocab.json (lookup/answer_labels.py) and regions_usa.json (synth/region_labels.py)")
    parser.add_argument("--real", type=Path, default=Path("/kaggle/input"), help="holds the real maps' records.jsonl")
    parser.add_argument("--model", default="Qwen/Qwen2.5-VL-3B-Instruct")
    parser.add_argument("--adapter", help="the LoRA adapter to continue from")
    parser.add_argument("--max-pixels", type=int, default=640_000)
    parser.add_argument("--head-minutes", type=float, default=40, help="stage A: the head alone")
    parser.add_argument("--head-maps", type=int, default=2000, help="stage A maps per source (US, India, MapQA)")
    parser.add_argument("--aux-weight", type=float, default=0.5, help="stage B: weight of the head's patch loss")
    parser.add_argument("--head-lr", type=float, default=1e-4, help="stage B: the head's learning rate")
    parser.add_argument("--head-only", action="store_true", help="stop after stage A and its report")
    parser.set_defaults(epochs=1, hours=4.5, lr=1e-4, batch_size=1, grad_accum=16, eval_batch_size=8,
                        max_new_tokens=600, india_eval_maps=300, synth_train_maps=2000, india_train_maps=2000)
    args = parser.parse_args()
    set_seed(args.seed)
    args.out.mkdir(parents=True, exist_ok=True)
    (args.out / "args.json").write_text(json.dumps(vars(args), default=str, indent=1))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype = torch.float32 if device.type == "cpu" else (
        torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16)
    amp_dtype = None if device.type == "cpu" else dtype
    labels = Labels(args.labels)
    processor = AutoProcessor.from_pretrained(args.model, max_pixels=args.max_pixels, min_pixels=28 * 28)
    processor.tokenizer.padding_side = "left"
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(args.model, torch_dtype=dtype,
                                                               attn_implementation="sdpa").to(device)
    model = load_adapter(model, find_saved(args.adapter, "adapter_config.json"), trainable=True)
    report_lora(model)
    wrapper = attach(model, len(labels.vocab), device)

    mapqa_root = find_images_root(args.mapqa, "map_*.png")
    usa_root = find_images_root(args.synth, "synth_*")
    india_root = find_images_root(args.india, "synthind_*")
    labelled_ids = set(json.loads(next(args.labels.rglob("mapqa_train_ids.json")).read_text()))
    mapqa_train = [r for r in read_split(args.data, "train", mapqa_root) if r["id"] in labelled_ids]
    mapqa_train = sample_records(mapqa_train, args.train_maps, args.seed)
    usa_train = sample_records(read_split(usa_root, "train", usa_root), args.synth_train_maps, args.seed)
    india_train = sample_records(read_split(india_root, "train", india_root), args.india_train_maps, args.seed)
    print(f"training maps: {len(mapqa_train)} MapQA + {len(usa_train)} US + {len(india_train)} India", flush=True)

    # stage A
    for p in model.parameters():
        p.requires_grad_(False)
    wrapper.head.requires_grad_(True)
    teach = (sample_records(mapqa_train, args.head_maps, 1) + sample_records(usa_train, args.head_maps, 1)
             + sample_records(india_train, args.head_maps, 1))
    model.eval()
    teach_head(wrapper, processor, teach, labels, device, amp_dtype, args.head_minutes, args.seed)
    torch.save(wrapper.head.state_dict(), args.out / "region_head_stage_a.pt")

    mapqa_test = sample_records(read_split(args.data, "test", mapqa_root), args.eval_maps, seed=0)
    india_test = sample_records(read_split(india_root, "test", india_root), args.india_eval_maps, seed=0)
    real_probe = [{"id": p.stem, "image_path": str(p)} for p in sorted(labels.real.glob("*.jpg"))
                  if (labels.folder / "real" / f"{p.stem}.png").exists()]
    if args.eval_limit:
        mapqa_test, india_test = mapqa_test[: args.eval_limit], india_test[: args.eval_limit]
        real_probe = real_probe[: args.eval_limit]
    head_sets = [("india_as_drawn", india_test[:150], False), ("india_moved", india_test[:150], True),
                 ("mapqa", mapqa_test[:150], False), ("real_india", real_probe, False)]
    report = {"stage_a": head_report(wrapper, processor, head_sets, labels, device, amp_dtype)}
    (args.out / "metrics_head.json").write_text(json.dumps(report, indent=1))
    if args.head_only:
        return

    # stage B
    lora = [p for n, p in model.named_parameters() if "lora_" in n]
    for p in lora:
        p.requires_grad_(True)
    model.train()
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    records = mapqa_train + usa_train + india_train
    loader = DataLoader(TrainSet(records, labels), batch_size=args.batch_size, shuffle=True, num_workers=args.workers,
                        collate_fn=Collate(processor, len(labels.vocab)))

    def forward(batch):
        enc, extra = batch
        wrapper.pending = extra
        loss = model(**enc.to(device)).loss
        if wrapper.aux_loss is not None:
            loss = loss + args.aux_weight * wrapper.aux_loss
        return loss

    groups = [{"params": lora}, {"params": list(wrapper.head.parameters()), "lr": args.head_lr}]
    print(f"{len(loader)} batches per epoch", flush=True)
    train_loop(model, loader, args, forward, args.out, amp_dtype, groups)
    save_adapter(model, args.out / "adapter")
    torch.save(wrapper.head.state_dict(), args.out / "region_head.pt")

    model.eval()
    model.config.use_cache = True
    if device.type == "cuda":
        torch.cuda.empty_cache()
    report["stage_b"] = head_report(wrapper, processor, head_sets, labels, device, amp_dtype)
    (args.out / "metrics_head.json").write_text(json.dumps(report, indent=1))

    real = real_records(args.real)[: args.eval_limit]
    if real:
        answers, countries = predict(model, wrapper, processor, real, labels, args, device, amp_dtype)
        with open(args.out / "predictions_real.jsonl", "w") as f:
            for record, answer, country in zip(real, answers, countries):
                f.write(json.dumps({"id": record["id"], "prediction": answer, "country": country}) + "\n")
    for name, records in (("india", india_test), ("mapqa", mapqa_test)):
        answers, countries = predict(model, wrapper, processor, records, labels, args, device, amp_dtype)
        evaluate_and_save(records, answers, args.out, name)
        (args.out / f"countries_{name}.json").write_text(json.dumps(dict(zip([r["id"] for r in records], countries))))
    if report["stage_b"]["india_as_drawn"]["country_right"] < 0.95:  # reading apart from naming the country
        print("head names the country wrongly too often; scoring India again with the country given", flush=True)
        answers, _ = predict(model, wrapper, processor, india_test, labels, args, device, amp_dtype, force="IN")
        evaluate_and_save(india_test, answers, args.out, "india_country_given")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile answer_v2.py
#!/usr/bin/env python3
"""Learned answer sheet v2: small regions are read from a zoomed view, and the model is trained that way.

v1 (train/answer_sheet.py) pools every region's token from the map's 14-px patches; Delhi and Rhode Island cover
about a fifth of one patch, so their tokens are mostly their neighbours. Zooming v1 without training failed for two
reasons (outputs/mapqa-derender-zoom-v1/SUMMARY.md): the head had never seen zoomed views, and the reader had never
seen tokens made from them. v2 trains both:

- Stage A (head alone): half the steps are 3x views around a small region, labelled from the same shapefile labels.
- Stage B (LoRA + head): every region smaller than `--small-mass` patches gets its token from a 3x view around it,
  pooled by the head in that view. The head keeps its patch labels as a side loss, on the views too.
- Reading: the head reads the map, every region it finds fewer than `--small-mass` patches of is looked at again in
  a 3x view around where it put that region, and the token is pooled there. Still one model: no fit, no OCR.

In training the view is centred on the region's true middle, moved a little at random; in reading, on the head's.
Continues from the v1 adapter and head.

    KAGGLE_API_TOKEN=... python kaggle/launch.py v2 --owner skini501 --tag v2 --dataset sartma/mapqa-s \
        --india-dataset sartma/synth-india-choropleth --extra-dataset skini501/answer-v2-inputs \
        --extra-dataset skini501/real-sheets --no-wait -- --adapter /kaggle/input
"""

import argparse
import json
import random
import time
from pathlib import Path

import cv2
import numpy as np
import torch
from peft import PeftModel
from PIL import Image
from torch.utils.data import DataLoader
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

import answer_sheet as A
from common import (add_common_args, autocast, evaluate_and_save, find_images_root, find_saved, read_split,
                    sample_records, set_seed, train_loop)
from train_qwen import report_lora


# ---------------------------------------------------------------- zoomed views

def zoom_box(size, cx, cy, side_share):
    width, height = size
    side = side_share * max(size)
    left = int(min(max(0, cx - side / 2), max(0, width - side)))
    top = int(min(max(0, cy - side / 2), max(0, height - side)))
    return left, top, int(min(width, left + side)), int(min(height, top + side))


def zoom_view(image, label, box, zoom):
    size = (max(28, round((box[2] - box[0]) * zoom)), max(28, round((box[3] - box[1]) * zoom)))
    view = image.crop(box).resize(size, Image.BICUBIC)
    crop = None
    if label is not None:
        crop = cv2.resize(np.ascontiguousarray(label[box[1]:box[3], box[0]:box[2]]), size, interpolation=cv2.INTER_NEAREST)
    return view, crop


def label_areas(label, max_pixels, classes):
    """Each class's area in 14-px patches, as the processor will see the image."""
    scale = min(1.0, max_pixels / label.size)
    return np.bincount(label.ravel(), minlength=classes) * scale / 196.0


def true_view(image, label, k, args, rng):
    """A zoomed view around class k's true middle, moved by up to `jitter` of the view's side."""
    ys, xs = np.nonzero(label == k)
    side = args.zoom_side * max(image.size)
    cx = xs.mean() + rng.uniform(-args.jitter, args.jitter) * side
    cy = ys.mean() + rng.uniform(-args.jitter, args.jitter) * side
    return zoom_view(image, label, zoom_box(image.size, cx, cy, args.zoom_side), args.zoom)


# ---------------------------------------------------------------- the model

class ZoomVisual(A.AnswerSheetVisual):
    """v1's tower, plus region tokens taken from zoomed views where `pending["views"]` gives them."""

    def forward(self, hidden_states, grid_thw, **kwargs):
        pending = self.pending
        views = pending.get("views") if pending else None
        view_losses = []
        if views:
            device = hidden_states.device
            with torch.no_grad():
                _, grids = self.patch_grids(views["pixel_values"].to(device).type(hidden_states.dtype),
                                            views["grid_thw"].to(device))
            override = pending.setdefault("override", {})
            for (m, j, k), grid, shares in zip(views["owner"], grids, views["labels"]):
                logits = self.head(grid)
                if shares is not None:
                    view_losses.append(-(shares.to(device) * logits.float().log_softmax(-1)).sum(-1).mean())
                weights = logits.float().softmax(-1).detach()[:, k:k + 1] ** self.sharpen
                override.setdefault(m, {})[j] = self.pool(grid, weights)[0]
        out = super().forward(hidden_states, grid_thw, **kwargs)
        if view_losses:
            losses = view_losses + ([self.aux_loss] if self.aux_loss is not None else [])
            self.aux_loss = torch.stack(losses).mean()
        return out


def attach(model, classes, device):
    for module in model.modules():
        if "visual" in module._modules and not isinstance(module._modules["visual"], A.AnswerSheetVisual):
            wrapper = ZoomVisual(module._modules["visual"], classes)
            wrapper.head.to(device=device, dtype=torch.float32)
            module._modules["visual"] = wrapper
            return wrapper
    raise SystemExit("could not find the vision tower")


# ---------------------------------------------------------------- training inputs

class TrainSet(A.TrainSet):
    """v1's examples, plus a zoomed view (and its labels) for every shown region smaller than `small_mass` patches."""

    def __init__(self, records, labels, args):
        super().__init__(records, labels)
        self.args = args

    def __getitem__(self, i):
        image, label, names, classes, target = super().__getitem__(i)
        views = []
        if label is not None:
            areas = label_areas(label, self.args.max_pixels, len(self.labels.vocab))
            rng = random.Random()
            small = [(j, k) for j, k in enumerate(classes) if 0 < areas[k] < self.args.small_mass]
            for j, k in rng.sample(small, min(len(small), self.args.max_views)):
                view, crop = true_view(image, label, k, self.args, rng)
                views.append((j, k, view, crop))
        return image, label, names, classes, target, views


class Collate(A.Collate):
    def __call__(self, batch):
        enc, extra = super().__call__([item[:5] for item in batch])
        owner, images, crops = [], [], []
        for m, item in enumerate(batch):
            for j, k, view, crop in item[5]:
                owner.append((m, j, k))
                images.append(view)
                crops.append(crop)
        if images:
            seen = self.processor.image_processor(images=images, return_tensors="pt")
            shares = [None if crop is None else A.patch_shares(crop, h, w, self.classes)
                      for crop, (_, h, w) in zip(crops, seen["image_grid_thw"].tolist())]
            extra["views"] = {"pixel_values": seen["pixel_values"], "grid_thw": seen["image_grid_thw"],
                              "owner": owner, "labels": shares}
        return enc, extra


# ---------------------------------------------------------------- stage A

def teach_head(wrapper, processor, records, labels, args, device, amp_dtype):
    """The head alone on frozen features: whole maps (half moved) and, `--zoom-share` of the time, zoomed views."""
    head = wrapper.head
    optimizer = torch.optim.AdamW(head.parameters(), lr=5e-4, weight_decay=0.01)
    deadline, began, step, window = time.time() + args.head_minutes * 60, time.time(), 0, {"map": [], "view": []}
    rng = random.Random(args.seed)
    head.train()
    while time.time() < deadline:
        for record in rng.sample(records, len(records)):
            image = Image.open(record["image_path"]).convert("RGB")
            label = labels.label(record, image.size)
            kind = "map"
            if rng.random() < args.zoom_share:
                areas = label_areas(label, args.max_pixels, len(labels.vocab))
                present = [k for k in range(1, len(labels.vocab)) if areas[k] > 0]
                small = [k for k in present if areas[k] < args.small_mass]
                image, label = true_view(image, label, rng.choice(small or present), args, rng)
                kind = "view"
            elif rng.random() < 0.5:
                image, label = A.move(image, label, rng)
            grid = A.map_grid(wrapper, processor, image, device, amp_dtype)
            shares = A.patch_shares(label, grid.shape[0], grid.shape[1], len(labels.vocab)).to(device)
            loss = -(shares * head(grid).log_softmax(-1)).sum(-1).mean()
            (loss / 4).backward()
            window[kind].append(loss.item())
            step += 1
            if step % 4 == 0:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
            if step % 200 == 0:
                print(json.dumps({"head_step": step, **{f"loss_{n}": round(float(np.mean(v)), 4) for n, v in window.items() if v},
                                  "minutes": round((time.time() - began) / 60, 1)}), flush=True)
                window = {"map": [], "view": []}
            if time.time() > deadline:
                break
    head.eval()
    print(f"head taught on {step} images", flush=True)


# ---------------------------------------------------------------- reading

def zoom_tokens(wrapper, processor, image, label, grid, chances, country, codes, labels, args, device, amp_dtype):
    """{position in codes: token} for the regions the head finds fewer than `small_mass` patches of, and, where
    labels exist, each one's purity (share of its token that is its own pixels) before and after."""
    rows, columns = grid.shape[:2]
    down, across = torch.meshgrid(torch.arange(rows, device=device), torch.arange(columns, device=device), indexing="ij")
    down, across = down.flatten().float(), across.flatten().float()
    shares = None if label is None else A.patch_shares(label, rows, columns, len(labels.vocab)).to(device)
    overrides, stats = {}, []
    for j, code in enumerate(codes):
        k = labels.index[f"{country}-{code}"]
        p = chances[:, k]
        if p.sum() >= args.small_mass or p.max() < args.min_peak:
            continue
        w = p ** 4
        cx = float((w * (across + 0.5)).sum() / w.sum()) / columns * image.width
        cy = float((w * (down + 0.5)).sum() / w.sum()) / rows * image.height
        view, crop = zoom_view(image, label, zoom_box(image.size, cx, cy, args.zoom_side), args.zoom)
        view_grid = A.map_grid(wrapper, processor, view, device, amp_dtype)
        with torch.no_grad():
            weights = wrapper.head(view_grid).float().softmax(-1)[:, k:k + 1] ** wrapper.sharpen
            with autocast(device, amp_dtype):
                overrides[j] = wrapper.pool(view_grid, weights)[0]
        if shares is not None and shares[:, k].sum() > 0:
            before = p ** wrapper.sharpen
            view_shares = A.patch_shares(crop, view_grid.shape[0], view_grid.shape[1], len(labels.vocab)).to(device)
            after = weights[:, 0] / weights.sum().clamp_min(1e-6)
            stats.append({"code": code, "before": float((before / before.sum() * shares[:, k]).sum()),
                          "after": float((after * view_shares[:, k]).sum())})
    return overrides, stats


def predict(model, wrapper, processor, records, labels, args, device, amp_dtype, zoom=True):
    answers, countries, stats, began = [], [], [], time.time()
    for start in range(0, len(records), args.eval_batch_size):
        batch = records[start: start + args.eval_batch_size]
        texts, images, classes, overrides = [], [], [], {}
        for m, record in enumerate(batch):
            image = Image.open(record["image_path"]).convert("RGB")
            grid = A.map_grid(wrapper, processor, image, device, amp_dtype)
            with torch.no_grad():
                chances = wrapper.head(grid).float().softmax(-1)
            country = A.country_of(chances.cpu(), labels.vocab)
            codes = labels.listed[country]
            if zoom:
                label = labels.label(record, image.size)
                overrides[m], found = zoom_tokens(wrapper, processor, image, label, grid, chances, country, codes,
                                                  labels, args, device, amp_dtype)
                stats += [{"id": record["id"], **s} for s in found]
            countries.append(country)
            texts.append(A.chat(processor, codes))
            images += [image] + [A.PLACEHOLDER] * len(codes)
            classes.append([labels.index[f"{country}-{c}"] for c in codes])
        enc = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)
        wrapper.pending = {"classes": classes, "labels": [None] * len(batch), "override": overrides}
        with torch.no_grad(), autocast(device, amp_dtype):
            ids = model.generate(**enc, max_new_tokens=args.max_new_tokens, do_sample=False, use_cache=True)
        answers += processor.batch_decode(ids[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"predicted {len(answers)}/{len(records)} ({(time.time() - began) / 60:.1f} min), "
              f"{sum(len(o) for o in overrides.values())} regions zoomed in this batch", flush=True)
    return answers, countries, stats


def summarise(stats):
    by = {}
    for s in stats:
        by.setdefault(s["code"], []).append(s)
    return {code: {"zoomed": len(v), "purity_before": float(np.median([s["before"] for s in v])),
                   "purity_after": float(np.median([s["after"] for s in v]))}
            for code, v in sorted(by.items(), key=lambda kv: -len(kv[1]))}


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    add_common_args(parser)
    parser.add_argument("--labels", type=Path, default=Path("/kaggle/input"))
    parser.add_argument("--real", type=Path, default=Path("/kaggle/input"))
    parser.add_argument("--model", default="Qwen/Qwen2.5-VL-3B-Instruct")
    parser.add_argument("--adapter", required=True, help="holds v1's adapter/ and region_head.pt")
    parser.add_argument("--max-pixels", type=int, default=640_000)
    parser.add_argument("--zoom", type=float, default=3.0)
    parser.add_argument("--zoom-side", type=float, default=0.12, help="view side, as a share of the map's longer side")
    parser.add_argument("--small-mass", type=float, default=3.0, help="zoom regions smaller than this many patches")
    parser.add_argument("--min-peak", type=float, default=0.05, help="in reading, skip regions the head barely sees")
    parser.add_argument("--jitter", type=float, default=0.15, help="in training, move the view by up to this share")
    parser.add_argument("--max-views", type=int, default=10, help="in training, zoomed views per map")
    parser.add_argument("--zoom-share", type=float, default=0.5, help="stage A: share of steps on zoomed views")
    parser.add_argument("--head-minutes", type=float, default=30)
    parser.add_argument("--head-maps", type=int, default=2000)
    parser.add_argument("--aux-weight", type=float, default=0.5)
    parser.add_argument("--head-lr", type=float, default=1e-4)
    parser.set_defaults(epochs=1, hours=4.5, lr=1e-4, batch_size=1, grad_accum=16, eval_batch_size=8,
                        max_new_tokens=600, india_eval_maps=300, synth_train_maps=2000, india_train_maps=2000)
    args = parser.parse_args()
    set_seed(args.seed)
    args.out.mkdir(parents=True, exist_ok=True)
    (args.out / "args.json").write_text(json.dumps(vars(args), default=str, indent=1))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype = torch.float32 if device.type == "cpu" else (
        torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16)
    amp_dtype = None if device.type == "cpu" else dtype
    labels = A.Labels(args.labels)
    processor = AutoProcessor.from_pretrained(args.model, max_pixels=args.max_pixels, min_pixels=28 * 28)
    processor.tokenizer.padding_side = "left"
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(args.model, torch_dtype=dtype,
                                                               attn_implementation="sdpa").to(device)
    model = A.load_adapter(model, find_saved(args.adapter, "adapter_config.json"), trainable=True)
    report_lora(model)
    wrapper = attach(model, len(labels.vocab), device)
    head_file = next(Path(args.adapter).rglob("region_head.pt"))
    wrapper.head.load_state_dict(torch.load(head_file, map_location=device))
    print(f"continuing from {head_file}", flush=True)

    mapqa_root = find_images_root(args.mapqa, "map_*.png")
    usa_root = find_images_root(args.synth, "synth_*")
    india_root = find_images_root(args.india, "synthind_*")
    labelled_ids = set(json.loads(next(args.labels.rglob("mapqa_train_ids.json")).read_text()))
    mapqa_train = [r for r in read_split(args.data, "train", mapqa_root) if r["id"] in labelled_ids]
    mapqa_train = sample_records(mapqa_train, args.train_maps, args.seed + 7)  # a different draw from v1's
    usa_train = sample_records(read_split(usa_root, "train", usa_root), args.synth_train_maps, args.seed + 7)
    india_train = sample_records(read_split(india_root, "train", india_root), args.india_train_maps, args.seed + 7)
    print(f"training maps: {len(mapqa_train)} MapQA + {len(usa_train)} US + {len(india_train)} India", flush=True)

    # stage A
    for p in model.parameters():
        p.requires_grad_(False)
    wrapper.head.requires_grad_(True)
    model.eval()
    teach = (sample_records(mapqa_train, args.head_maps, 1) + sample_records(usa_train, args.head_maps, 1)
             + sample_records(india_train, args.head_maps, 1))
    teach_head(wrapper, processor, teach, labels, args, device, amp_dtype)
    torch.save(wrapper.head.state_dict(), args.out / "region_head_stage_a.pt")

    mapqa_test = sample_records(read_split(args.data, "test", mapqa_root), args.eval_maps, seed=0)
    india_test = sample_records(read_split(india_root, "test", india_root), args.india_eval_maps, seed=0)
    real_probe = [{"id": p.stem, "image_path": str(p)} for p in sorted(labels.real.glob("*.jpg"))
                  if (labels.folder / "real" / f"{p.stem}.png").exists()]
    if args.eval_limit:
        mapqa_test, india_test, real_probe = (x[: args.eval_limit] for x in (mapqa_test, india_test, real_probe))
    head_sets = [("india_as_drawn", india_test[:150], False), ("india_moved", india_test[:150], True),
                 ("mapqa", mapqa_test[:150], False), ("real_india", real_probe, False)]
    report = {"stage_a": A.head_report(wrapper, processor, head_sets, labels, device, amp_dtype)}
    (args.out / "metrics_head.json").write_text(json.dumps(report, indent=1))

    # stage B
    lora = [p for n, p in model.named_parameters() if "lora_" in n]
    for p in lora:
        p.requires_grad_(True)
    model.train()
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    loader = DataLoader(TrainSet(mapqa_train + usa_train + india_train, labels, args), batch_size=args.batch_size,
                        shuffle=True, num_workers=args.workers, collate_fn=Collate(processor, len(labels.vocab)))

    def forward(batch):
        enc, extra = batch
        wrapper.pending = extra
        loss = model(**enc.to(device)).loss
        if wrapper.aux_loss is not None:
            loss = loss + args.aux_weight * wrapper.aux_loss
        return loss

    groups = [{"params": lora}, {"params": list(wrapper.head.parameters()), "lr": args.head_lr}]
    print(f"{len(loader)} batches per epoch", flush=True)
    train_loop(model, loader, args, forward, args.out, amp_dtype, groups)
    A.save_adapter(model, args.out / "adapter")
    torch.save(wrapper.head.state_dict(), args.out / "region_head.pt")

    model.eval()
    model.config.use_cache = True
    if device.type == "cuda":
        torch.cuda.empty_cache()
    report["stage_b"] = A.head_report(wrapper, processor, head_sets, labels, device, amp_dtype)
    (args.out / "metrics_head.json").write_text(json.dumps(report, indent=1))

    sets = [("real", A.real_records(args.real)[: args.eval_limit]), ("mapqa", mapqa_test), ("india", india_test)]
    for name, records in sets:
        print(f"scoring {len(records)} {name} maps with zoom", flush=True)
        answers, countries, stats = predict(model, wrapper, processor, records, labels, args, device, amp_dtype)
        (args.out / f"purity_{name}.json").write_text(json.dumps(summarise(stats), indent=1))
        if name == "real":
            with open(args.out / "predictions_real.jsonl", "w") as f:
                for record, answer, country in zip(records, answers, countries):
                    f.write(json.dumps({"id": record["id"], "prediction": answer, "country": country}) + "\n")
        else:
            evaluate_and_save(records, answers, args.out, name)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile answer_v3.py
#!/usr/bin/env python3
"""Answer sheet v3: find regions on an S2 (2x) patch grid, and zoom each small region by as much as it needs.

v2 (train/answer_v2.py, outputs/mapqa-derender-v2-zoom/SUMMARY.md) zoomed small regions 3x and fixed many of them,
but its own records show three gaps:
- finding: the head picks where to zoom from the 1x grid, where Delhi is a fifth of a patch. Below a 5% peak it
  skips the region, which happened for Delhi on 58 of 68 real maps and Rhode Island on 169 of 400 MapQA maps
- too little zoom: at 3x, Rhode Island, Delhi and Andaman tokens are still only 56-69% their own pixels
- zooming what is not small: on real maps medium states (Nagaland, Punjab, Himachal) got zoomed and their tokens
  got worse (Himachal 0.74 -> 0.23); about 10 swapped tokens per map is the likely cause of whole-map class shifts

v3, two modes (--mode):
- s2: "scaling on scales" as region tokens. The map is enlarged 2x, cut into 2x2 tiles of the map's own size, each
  run through the tower, and the patch grids stitched into one grid at twice the detail (Delhi: ~0.8 patch). The
  head reads that grid and every region's token is pooled from it. No zoom. Unlike the S2 tried on 2026-09-28
  (train_qwen.py --s2), nothing is averaged back onto the coarse grid and no new weights sit between tower and LLM:
  pooling is per region and goes through Qwen's own merger.
- zoom (default): s2, plus every region the head finds less than `--small-mass` (1x) patches of is re-read in a
  view zoomed so the region covers about `--target-patches` patches (2x-8x). The zoomed token replaces the S2 token
  only when the head is more confident about the region in the view (mean belief over the pooled patches).

The language model still sees the 1x map tokens; only the region tokens change. Continues from v2's weights.

    KAGGLE_API_TOKEN=... python kaggle/launch.py v3 --owner skini501 --tag zoom \\
        --india-dataset sartma/synth-india-choropleth --extra-dataset skini501/answer-v2-inputs \\
        --extra-dataset skini501/real-sheets --from-run skini501/mapqa-derender-v2-zoom --no-wait \\
        -- --adapter /kaggle/input --prefer v2-zoom --mode zoom
"""

import argparse
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

import answer_sheet as A
import answer_v2 as Z
from common import (add_common_args, autocast, evaluate_and_save, find_images_root, read_split, sample_records,
                    set_seed, train_loop)
from train_qwen import report_lora

SCALE = 2


# ---------------------------------------------------------------- S2 grid

def s2_tiles(image, scale=SCALE):
    """The image enlarged `scale` times and cut into scale x scale tiles of the image's own size, row by row."""
    big = image.resize((image.width * scale, image.height * scale), Image.BICUBIC)
    return [big.crop((c * image.width, r * image.height, (c + 1) * image.width, (r + 1) * image.height))
            for r in range(scale) for c in range(scale)]


def stitch(grids, scale=SCALE):
    """scale * scale tile grids (rows, columns, width), row by row -> one grid."""
    return torch.cat([torch.cat(grids[r * scale:(r + 1) * scale], dim=1) for r in range(scale)], dim=0)


def fine_grid(wrapper, processor, image, device, amp_dtype):
    seen = processor.image_processor(images=s2_tiles(image), return_tensors="pt")
    with torch.no_grad(), autocast(device, amp_dtype):
        _, grids = wrapper.patch_grids(seen["pixel_values"].to(device).type(wrapper.visual.dtype),
                                       seen["image_grid_thw"].to(device))
    return stitch(grids)


def belief(chances, k, sharpen):
    """The head's mean belief in class k over the patches its token would be pooled from."""
    p = chances[:, k]
    w = p ** sharpen
    return float((w * p).sum() / w.sum().clamp_min(1e-6))


def zoom_for(area, args):
    """Zoom so a region of `area` (1x) patches covers about `--target-patches` patches."""
    return float(np.clip(math.sqrt(args.target_patches / max(area, 0.02)), args.min_zoom, args.max_zoom))


def view_around(image, label, cx, cy, zoom, args):
    """A view zoomed `zoom` times around (cx, cy); its side shrinks as zoom grows so views stay one size."""
    side_share = args.view_extent / zoom
    return Z.zoom_view(image, label, Z.zoom_box(image.size, cx, cy, side_share), zoom)


# ---------------------------------------------------------------- the model

class S2Visual(A.AnswerSheetVisual):
    """Region tokens from the S2 grid (`pending["fine"]`), and from zoomed views where `pending["views"]` gives
    them. The 1x map still goes to the language model as before."""

    def forward(self, hidden_states, grid_thw, **kwargs):
        pending = self.pending
        losses = []
        if pending:
            device = hidden_states.device
            override = pending.setdefault("override", {})
            views = pending.get("views")
            if views:
                with torch.no_grad():
                    _, grids = self.patch_grids(views["pixel_values"].to(device).type(hidden_states.dtype),
                                                views["grid_thw"].to(device))
                for (m, j, k), grid, shares in zip(views["owner"], grids, views["labels"]):
                    logits = self.head(grid)
                    if shares is not None:
                        losses.append(-(shares.to(device) * logits.float().log_softmax(-1)).sum(-1).mean())
                    weights = logits.float().softmax(-1).detach()[:, k:k + 1] ** self.sharpen
                    override.setdefault(m, {})[j] = self.pool(grid, weights)[0]
            fine = pending.get("fine")
            if fine:
                with torch.no_grad():
                    _, grids = self.patch_grids(fine["pixel_values"].to(device).type(hidden_states.dtype),
                                                fine["grid_thw"].to(device))
                for m, classes in enumerate(pending["classes"]):
                    grid = stitch(grids[m * SCALE ** 2:(m + 1) * SCALE ** 2])
                    logits = self.head(grid)
                    shares = fine["labels"][m]
                    if shares is not None:
                        losses.append(-(shares.to(device) * logits.float().log_softmax(-1)).sum(-1).mean())
                    regions = self.region_tokens(grid, logits, classes)
                    for j in range(len(classes)):
                        override.setdefault(m, {}).setdefault(j, regions[j])  # zoomed tokens win
        out = super().forward(hidden_states, grid_thw, **kwargs)
        if losses:
            self.aux_loss = torch.stack(losses + ([self.aux_loss] if self.aux_loss is not None else [])).mean()
        return out


def attach(model, classes, device):
    for module in model.modules():
        if "visual" in module._modules and not isinstance(module._modules["visual"], A.AnswerSheetVisual):
            wrapper = S2Visual(module._modules["visual"], classes)
            wrapper.head.to(device=device, dtype=torch.float32)
            module._modules["visual"] = wrapper
            return wrapper
    raise SystemExit("could not find the vision tower")


# ---------------------------------------------------------------- training inputs

def true_centre(label, k):
    ys, xs = np.nonzero(label == k)
    return xs.mean(), ys.mean()


class TrainSet(A.TrainSet):
    """v1's examples, plus S2 tiles for every map and (mode zoom) a sized view for every small shown region."""

    def __init__(self, records, labels, args):
        super().__init__(records, labels)
        self.args = args

    def __getitem__(self, i):
        image, label, names, classes, target = super().__getitem__(i)
        views = []
        if label is not None and self.args.mode == "zoom":
            areas = Z.label_areas(label, self.args.max_pixels, len(self.labels.vocab))
            rng = random.Random()
            small = [(j, k) for j, k in enumerate(classes) if 0 < areas[k] < self.args.small_mass]
            for j, k in rng.sample(small, min(len(small), self.args.max_views)):
                zoom = float(np.clip(zoom_for(areas[k], self.args) * rng.uniform(0.8, 1.25),
                                     self.args.min_zoom, self.args.max_zoom))
                cx, cy = true_centre(label, k)
                side = self.args.view_extent / zoom * max(image.size)
                cx += rng.uniform(-self.args.jitter, self.args.jitter) * side
                cy += rng.uniform(-self.args.jitter, self.args.jitter) * side
                view, crop = view_around(image, label, cx, cy, zoom, self.args)
                views.append((j, k, view, crop))
        return image, label, names, classes, target, views


class Collate(Z.Collate):
    def __call__(self, batch):
        enc, extra = super().__call__(batch)
        tiles = [tile for item in batch for tile in s2_tiles(item[0])]
        seen = self.processor.image_processor(images=tiles, return_tensors="pt")
        thw = seen["image_grid_thw"].tolist()
        shares = []
        for m, item in enumerate(batch):
            _, h, w = thw[m * SCALE ** 2]
            shares.append(None if item[1] is None else A.patch_shares(item[1], h * SCALE, w * SCALE, self.classes))
        extra["fine"] = {"pixel_values": seen["pixel_values"], "grid_thw": seen["image_grid_thw"], "labels": shares}
        return enc, extra


# ---------------------------------------------------------------- stage A

def teach_head(wrapper, processor, records, labels, args, device, amp_dtype):
    """The head alone on frozen features: S2 grids of whole maps (half moved) and, `--zoom-share` of the time in
    mode zoom, sized views around a small region."""
    head = wrapper.head
    optimizer = torch.optim.AdamW(head.parameters(), lr=5e-4, weight_decay=0.01)
    deadline, began, step, window = time.time() + args.head_minutes * 60, time.time(), 0, {"s2": [], "view": []}
    rng = random.Random(args.seed)
    zoom_share = args.zoom_share if args.mode == "zoom" else 0.0
    head.train()
    while time.time() < deadline:
        for record in rng.sample(records, len(records)):
            image = Image.open(record["image_path"]).convert("RGB")
            label = labels.label(record, image.size)
            if rng.random() < zoom_share:
                areas = Z.label_areas(label, args.max_pixels, len(labels.vocab))
                present = [k for k in range(1, len(labels.vocab)) if areas[k] > 0]
                small = [k for k in present if areas[k] < args.small_mass]
                k = rng.choice(small or present)
                zoom = float(np.clip(zoom_for(areas[k], args) * rng.uniform(0.8, 1.25), args.min_zoom, args.max_zoom))
                cx, cy = true_centre(label, k)
                side = args.view_extent / zoom * max(image.size)
                view, crop = view_around(image, label, cx + rng.uniform(-args.jitter, args.jitter) * side,
                                         cy + rng.uniform(-args.jitter, args.jitter) * side, zoom, args)
                grid, target, kind = A.map_grid(wrapper, processor, view, device, amp_dtype), crop, "view"
            else:
                if rng.random() < 0.5:
                    image, label = A.move(image, label, rng)
                grid, target, kind = fine_grid(wrapper, processor, image, device, amp_dtype), label, "s2"
            shares = A.patch_shares(target, grid.shape[0], grid.shape[1], len(labels.vocab)).to(device)
            loss = -(shares * head(grid).log_softmax(-1)).sum(-1).mean()
            (loss / 4).backward()
            window[kind].append(loss.item())
            step += 1
            if step % 4 == 0:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
            if step % 200 == 0:
                print(json.dumps({"head_step": step, **{f"loss_{n}": round(float(np.mean(v)), 4)
                                                         for n, v in window.items() if v},
                                  "minutes": round((time.time() - began) / 60, 1)}), flush=True)
                window = {"s2": [], "view": []}
            if time.time() > deadline:
                break
    head.eval()
    print(f"head taught on {step} images", flush=True)


# ---------------------------------------------------------------- head report: finding and purity, 1x against S2

def head_report(wrapper, processor, sets, labels, args, device, amp_dtype):
    """Per set: country right, patch accuracy, token purity on the S2 grid, and for each small region how often
    the head finds it at all (peak >= --min-peak) on the 1x grid against the S2 grid."""
    report = {}
    for name, records in sets:
        right_country, patch_right, patches = 0, 0, 0
        purity = {"big": [], "small": []}
        found = {}
        for record in records:
            image = Image.open(record["image_path"]).convert("RGB")
            label = labels.label(record, image.size)
            if label is None:
                continue
            country = A.Labels.country(record)
            coarse = A.map_grid(wrapper, processor, image, device, amp_dtype)
            grid = fine_grid(wrapper, processor, image, device, amp_dtype)
            with torch.no_grad():
                chances = wrapper.head(grid).float().softmax(-1).cpu()
                coarse_chances = wrapper.head(coarse).float().softmax(-1).cpu()
            shares = A.patch_shares(label, grid.shape[0], grid.shape[1], len(labels.vocab))
            right_country += A.country_of(chances, labels.vocab) == country
            on_map = shares[:, 1:].sum(1) > 0.5
            patch_right += int((chances[on_map].argmax(1) == shares[on_map].argmax(1)).sum())
            patches += int(on_map.sum())
            for code in labels.listed[country]:
                k = labels.index[f"{country}-{code}"]
                if shares[:, k].sum() == 0:
                    continue
                weights = chances[:, k] ** wrapper.sharpen
                weights = weights / weights.sum().clamp_min(1e-6)
                small = code in A.SMALL[country]
                purity["small" if small else "big"].append(float((weights * shares[:, k]).sum()))
                if small:
                    hit = found.setdefault(code, {"maps": 0, "found_1x": 0, "found_s2": 0})
                    hit["maps"] += 1
                    hit["found_1x"] += float(coarse_chances[:, k].max()) >= args.min_peak
                    hit["found_s2"] += float(chances[:, k].max()) >= args.min_peak
        report[name] = {"maps": len(records), "country_right": right_country / max(1, len(records)),
                        "patch_accuracy": patch_right / max(1, patches),
                        **{f"{size}_purity_median": float(np.median(v)) if v else None for size, v in purity.items()},
                        "found": found}
        print(name, json.dumps(report[name]), flush=True)
    return report


# ---------------------------------------------------------------- reading

def region_overrides(wrapper, processor, image, label, grid, chances, country, codes, labels, args, device,
                     amp_dtype):
    """{position in codes: token} for every region (S2 grid, or a zoomed view where that helps), and per small
    region what happened, with purity where labels exist."""
    rows, columns = grid.shape[:2]
    down, across = torch.meshgrid(torch.arange(rows, device=device), torch.arange(columns, device=device), indexing="ij")
    down, across = down.flatten().float(), across.flatten().float()
    shares = None if label is None else A.patch_shares(label, rows, columns, len(labels.vocab)).to(device)
    classes = [labels.index[f"{country}-{c}"] for c in codes]
    with torch.no_grad(), autocast(device, amp_dtype):
        tokens = wrapper.pool(grid, chances[:, classes] ** wrapper.sharpen)
    overrides = {j: tokens[j] for j in range(len(codes))}
    stats = []
    if args.mode != "zoom":
        return overrides, stats
    for j, (code, k) in enumerate(zip(codes, classes)):
        p = chances[:, k]
        area = float(p.sum()) / SCALE ** 2
        if area >= args.small_mass:
            continue
        stat = {"code": code, "found": float(p.max()) >= args.min_peak, "kept_zoom": False}
        if stat["found"]:
            w = p ** 4
            cx = float((w * (across + 0.5)).sum() / w.sum()) / columns * image.width
            cy = float((w * (down + 0.5)).sum() / w.sum()) / rows * image.height
            zoom = zoom_for(area, args)
            view, crop = view_around(image, label, cx, cy, zoom, args)
            view_grid = A.map_grid(wrapper, processor, view, device, amp_dtype)
            with torch.no_grad():
                view_chances = wrapper.head(view_grid).float().softmax(-1)
            stat.update(zoom=round(zoom, 2), belief_s2=belief(chances, k, wrapper.sharpen),
                        belief_zoom=belief(view_chances, k, wrapper.sharpen))
            if stat["belief_zoom"] > stat["belief_s2"]:
                with torch.no_grad(), autocast(device, amp_dtype):
                    overrides[j] = wrapper.pool(view_grid, view_chances[:, k:k + 1] ** wrapper.sharpen)[0]
                stat["kept_zoom"] = True
            if crop is not None:
                view_shares = A.patch_shares(crop, view_grid.shape[0], view_grid.shape[1], len(labels.vocab))
                weights = view_chances[:, k] ** wrapper.sharpen
                stat["purity_zoom"] = float((weights / weights.sum().clamp_min(1e-6) * view_shares[:, k].to(device)).sum())
        if shares is not None and shares[:, k].sum() > 0:
            weights = p ** wrapper.sharpen
            stat["purity_s2"] = float((weights / weights.sum().clamp_min(1e-6) * shares[:, k]).sum())
        stats.append(stat)
    return overrides, stats


def predict(model, wrapper, processor, records, labels, args, device, amp_dtype):
    answers, countries, stats, began = [], [], [], time.time()
    for start in range(0, len(records), args.eval_batch_size):
        batch = records[start: start + args.eval_batch_size]
        texts, images, classes, overrides = [], [], [], {}
        for m, record in enumerate(batch):
            image = Image.open(record["image_path"]).convert("RGB")
            grid = fine_grid(wrapper, processor, image, device, amp_dtype)
            with torch.no_grad():
                chances = wrapper.head(grid).float().softmax(-1)
            country = A.country_of(chances.cpu(), labels.vocab)
            codes = labels.listed[country]
            label = labels.label(record, image.size)
            overrides[m], found = region_overrides(wrapper, processor, image, label, grid, chances, country, codes,
                                                   labels, args, device, amp_dtype)
            stats += [{"id": record["id"], **s} for s in found]
            countries.append(country)
            texts.append(A.chat(processor, codes))
            images += [image] + [A.PLACEHOLDER] * len(codes)
            classes.append([labels.index[f"{country}-{c}"] for c in codes])
        enc = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)
        wrapper.pending = {"classes": classes, "labels": [None] * len(batch), "override": overrides}
        with torch.no_grad(), autocast(device, amp_dtype):
            ids = model.generate(**enc, max_new_tokens=args.max_new_tokens, do_sample=False, use_cache=True)
        answers += processor.batch_decode(ids[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"predicted {len(answers)}/{len(records)} ({(time.time() - began) / 60:.1f} min)", flush=True)
    return answers, countries, stats


def summarise(stats):
    by = {}
    for s in stats:
        by.setdefault(s["code"], []).append(s)

    def median(v, key):
        v = [s[key] for s in v if s.get(key) is not None]
        return float(np.median(v)) if v else None

    return {code: {"small_on_map": len(v), "found": sum(s["found"] for s in v),
                   "kept_zoom": sum(s["kept_zoom"] for s in v), "zoom_median": median(v, "zoom"),
                   "purity_s2": median(v, "purity_s2"), "purity_zoom": median(v, "purity_zoom")}
            for code, v in sorted(by.items(), key=lambda kv: -len(kv[1]))}


# ---------------------------------------------------------------- main

def pick(root, marker, prefer):
    """The one folder under root holding `marker` whose path contains `prefer`."""
    hits = [p for p in Path(root).rglob(marker) if prefer in str(p)]
    if len(hits) != 1:
        raise SystemExit(f"want one {marker} under {root} with '{prefer}' in its path, found {[str(h) for h in hits]}")
    return hits[0]


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    add_common_args(parser)
    parser.add_argument("--labels", type=Path, default=Path("/kaggle/input"))
    parser.add_argument("--real", type=Path, default=Path("/kaggle/input"))
    parser.add_argument("--model", default="Qwen/Qwen2.5-VL-3B-Instruct")
    parser.add_argument("--adapter", required=True, help="holds the adapter/ and region_head.pt to continue from")
    parser.add_argument("--prefer", default="", help="pick the adapter and head whose path contains this")
    parser.add_argument("--mode", choices=["s2", "zoom"], default="zoom")
    parser.add_argument("--max-pixels", type=int, default=640_000)
    parser.add_argument("--small-mass", type=float, default=3.0, help="zoom regions smaller than this many 1x patches")
    parser.add_argument("--target-patches", type=float, default=8.0, help="zoom so a small region covers this many")
    parser.add_argument("--min-zoom", type=float, default=2.0)
    parser.add_argument("--max-zoom", type=float, default=8.0)
    parser.add_argument("--view-extent", type=float, default=0.36,
                        help="a view's side after zooming, as a share of the map's longer side (v2: 0.12 x 3)")
    parser.add_argument("--min-peak", type=float, default=0.02, help="in reading, skip regions the head barely sees")
    parser.add_argument("--jitter", type=float, default=0.15)
    parser.add_argument("--max-views", type=int, default=10)
    parser.add_argument("--zoom-share", type=float, default=0.4)
    parser.add_argument("--head-minutes", type=float, default=30)
    parser.add_argument("--head-maps", type=int, default=2000)
    parser.add_argument("--report-maps", type=int, default=100)
    parser.add_argument("--aux-weight", type=float, default=0.5)
    parser.add_argument("--head-lr", type=float, default=1e-4)
    parser.set_defaults(epochs=1, hours=3.75, lr=1e-4, batch_size=1, grad_accum=16, eval_batch_size=8,
                        max_new_tokens=600, india_eval_maps=300, synth_train_maps=2000, india_train_maps=2000)
    args = parser.parse_args()
    set_seed(args.seed)
    args.out.mkdir(parents=True, exist_ok=True)
    (args.out / "args.json").write_text(json.dumps(vars(args), default=str, indent=1))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype = torch.float32 if device.type == "cpu" else (
        torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16)
    amp_dtype = None if device.type == "cpu" else dtype
    labels = A.Labels(args.labels)
    processor = AutoProcessor.from_pretrained(args.model, max_pixels=args.max_pixels, min_pixels=28 * 28)
    processor.tokenizer.padding_side = "left"
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(args.model, torch_dtype=dtype,
                                                               attn_implementation="sdpa").to(device)
    adapter = pick(args.adapter, "adapter_config.json", args.prefer).parent
    head_file = pick(args.adapter, "region_head.pt", args.prefer)
    model = A.load_adapter(model, adapter, trainable=True)
    report_lora(model)
    wrapper = attach(model, len(labels.vocab), device)
    wrapper.head.load_state_dict(torch.load(head_file, map_location=device))
    print(f"mode {args.mode}; continuing from {adapter} and {head_file}", flush=True)

    mapqa_root = find_images_root(args.mapqa, "map_*.png")
    usa_root = find_images_root(args.synth, "synth_*")
    india_root = find_images_root(args.india, "synthind_*")
    labelled_ids = set(json.loads(next(args.labels.rglob("mapqa_train_ids.json")).read_text()))
    mapqa_train = [r for r in read_split(args.data, "train", mapqa_root) if r["id"] in labelled_ids]
    mapqa_train = sample_records(mapqa_train, args.train_maps, args.seed + 11)  # a draw unlike v1's and v2's
    usa_train = sample_records(read_split(usa_root, "train", usa_root), args.synth_train_maps, args.seed + 11)
    india_train = sample_records(read_split(india_root, "train", india_root), args.india_train_maps, args.seed + 11)
    print(f"training maps: {len(mapqa_train)} MapQA + {len(usa_train)} US + {len(india_train)} India", flush=True)

    mapqa_test = sample_records(read_split(args.data, "test", mapqa_root), args.eval_maps, seed=0)
    india_test = sample_records(read_split(india_root, "test", india_root), args.india_eval_maps, seed=0)
    real_probe = [{"id": p.stem, "image_path": str(p)} for p in sorted(labels.real.glob("*.jpg"))
                  if (labels.folder / "real" / f"{p.stem}.png").exists()]
    if args.eval_limit:
        mapqa_test, india_test, real_probe = (x[: args.eval_limit] for x in (mapqa_test, india_test, real_probe))
    head_sets = [("india", india_test[: args.report_maps]), ("mapqa", mapqa_test[: args.report_maps]),
                 ("real_india", real_probe)]
    for p in model.parameters():
        p.requires_grad_(False)
    model.eval()
    report = {"start": head_report(wrapper, processor, head_sets, labels, args, device, amp_dtype)}
    (args.out / "metrics_head.json").write_text(json.dumps(report, indent=1))

    # stage A
    wrapper.head.requires_grad_(True)
    teach = (sample_records(mapqa_train, args.head_maps, 1) + sample_records(usa_train, args.head_maps, 1)
             + sample_records(india_train, args.head_maps, 1))
    teach_head(wrapper, processor, teach, labels, args, device, amp_dtype)
    torch.save(wrapper.head.state_dict(), args.out / "region_head_stage_a.pt")
    report["stage_a"] = head_report(wrapper, processor, head_sets, labels, args, device, amp_dtype)
    (args.out / "metrics_head.json").write_text(json.dumps(report, indent=1))

    # stage B
    lora = [p for n, p in model.named_parameters() if "lora_" in n]
    for p in lora:
        p.requires_grad_(True)
    model.train()
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    loader = DataLoader(TrainSet(mapqa_train + usa_train + india_train, labels, args), batch_size=args.batch_size,
                        shuffle=True, num_workers=args.workers, collate_fn=Collate(processor, len(labels.vocab)))

    def forward(batch):
        enc, extra = batch
        wrapper.pending = extra
        loss = model(**enc.to(device)).loss
        if wrapper.aux_loss is not None:
            loss = loss + args.aux_weight * wrapper.aux_loss
        return loss

    groups = [{"params": lora}, {"params": list(wrapper.head.parameters()), "lr": args.head_lr}]
    print(f"{len(loader)} batches per epoch", flush=True)
    train_loop(model, loader, args, forward, args.out, amp_dtype, groups)
    A.save_adapter(model, args.out / "adapter")
    torch.save(wrapper.head.state_dict(), args.out / "region_head.pt")

    model.eval()
    model.config.use_cache = True
    if device.type == "cuda":
        torch.cuda.empty_cache()
    report["stage_b"] = head_report(wrapper, processor, head_sets, labels, args, device, amp_dtype)
    (args.out / "metrics_head.json").write_text(json.dumps(report, indent=1))

    sets = [("real", A.real_records(args.real)[: args.eval_limit]), ("mapqa", mapqa_test), ("india", india_test)]
    for name, records in sets:
        print(f"scoring {len(records)} {name} maps, mode {args.mode}", flush=True)
        answers, countries, stats = predict(model, wrapper, processor, records, labels, args, device, amp_dtype)
        (args.out / f"purity_{name}.json").write_text(json.dumps(summarise(stats), indent=1))
        if name == "real":
            with open(args.out / "predictions_real.jsonl", "w") as f:
                for record, answer, country in zip(records, answers, countries):
                    f.write(json.dumps({"id": record["id"], "prediction": answer, "country": country}) + "\n")
        else:
            evaluate_and_save(records, answers, args.out, name)


if __name__ == "__main__":
    main()


In [ ]:
run("python convert_mapqa.py --mapqa /kaggle/input --out /kaggle/working/data")

In [ ]:
run('python answer_v3.py --data /kaggle/working/data --mapqa /kaggle/input --out /kaggle/working/v3 --synth /kaggle/input --india /kaggle/input --adapter /kaggle/input --prefer v2-zoom --mode s2')